# `C13` — Anonymity and information visibility

Scenario **C13** of the Track A certification ladder (card `OMIE-CERT-C13-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C13). Oracle kind: **exact-parametric, carried by a metamorphic twin**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. A *world* is one tape; a *twin* is a second tape built from the first by a rule stated in section 4. *Bit-identical* means equal byte for byte (dtype, shape and every bit of every value). Every engine-side assertion is an **exact integer equality** on order ids, agent ids, prices in ticks (0.01 EUR/MWh) and quantities in ticks (0.1 MW); the participant view is compared **bit for bit between two runs in the same mode**, never against a euro figure.

## 1. Purpose

**Purpose.** OMIE Rules §45.2.12 require the order book to be displayed on an anonymous basis, and §46.1 restricts trade notification to the agent's own trades. This is not decoration. It bounds what a participant trading through the `Session` is *allowed* to see. A leak here would be the most insidious defect in the whole system: any strategy, study or learned policy built on the simulator could look skilful while it was in fact reading information that no real participant can see.

The sharpest available formulation is metamorphic rather than structural. Construct a book containing a foreign iceberg's hidden remainder and counterparty identifiers. Then assert that the participant view is **bit-identical** to the view from a counterfactual book in which that hidden remainder is absent and the counterparty identifiers are permuted. The participant view is everything `Session.top_of_book` and the anonymous book `get_anonymous_lob` show, plus the participant's fill reports, done flags, own trades and own orders. If the view cannot tell those two worlds apart, nothing in it can be leaking. Crucially the assertion must be **side-conditioned on ownership**, not blanket: §7.3 and §9.2 explicitly permit an agent to see *its own* full iceberg quantity, so a blanket hidden-quantity-invisibility claim would be wrong.

*(Adapted from `scenario-ladder.md` §C13, retargeted from the RL observation to the `Session` participant view; the ladder's own text still names the RL observation and Track B's L0 leak control; amending it, as a dated amendment, is for the ladder's owner.)*

**Release adaptation (REL-09).** The release has no RL observation, reward or action space. C13 used to certify that the RL agent's observation, reward and done flag could not carry counterparty identity or a foreign iceberg's hidden remainder. It now certifies the same for the **participant view** a `Session` hands its participant: `Session.top_of_book` and `get_anonymous_lob` after reset and after every step, the per-step `FillReport` and `done` flag, and at the end of the episode its own trades and own resting orders (`get_agent_trades` / `get_agent_orders` for its own id). The random RL actions become random `AgentOrder`s that also place the participant's own icebergs and FOK orders (section 4, step 5). The claim text, the new quotes, the action distribution and findings F22–F25 come from the domain analyst's retargeted claim. Everything that does not depend on the RL observation is unchanged: the engine twin, the trade-report partition and twin, the ownership side-condition at engine level, the premise check and rule-implied display, the positive control, the battery, capacity, coverage and eager checks. In the re-run history and findings below, "the RL agent", "the observation" and "the reward" read as "the participant", "the participant view" and the `FillReport`.

**Certifies:** §7.1 anonymous order book; §7.2 trade notification; §7.3 simulator modelling rules; §9.2 rows "LOB per contract", "Best bid / best ask", "Own orders", "Trade book".

**Known result** (`scenario-ladder.md` §C13, amended 2026-10-16 and 2026-10-20, retargeted to the participant view): the participant view is bit-identical across the two worlds wherever no tape iceberg fills a resting order of the participant on entry (the premise). The view is `top_of_book` and `get_anonymous_lob` after reset and after every step, plus the `FillReport` and the done flag of every step, plus the participant's own trades and own orders at the end. Where the premise fails, the two worlds differ exactly as §4.2.1 Rule 2 (1), (4) imply, with `A`'s display $v_1 = \min(P - u, R)$, $u = M - \lfloor M/P \rfloor P$ and $M$ the iceberg's swept quantity on entry, the participant's fill included. On this construction that difference is zero, so the view stays bit-identical on every step (section 5 (2r)). The view shows a foreign iceberg only by its visible quantity and names no foreign order id or trader id. Own-iceberg full quantity **is** visible to its owner: at engine level, and on the participant's own-orders reader for the participant's own icebergs. `get_agent_trades(a)` returns exactly $a$'s fills with no counterparty identifier, and the union over agents reconstructs the trade tape without duplication.

**Pass** (`scenario-ladder.md` §C13, amended, retargeted): the participant view is bit-identical where the premise holds. Where it fails, the rule-implied difference holds exactly, computed with $v_1 = \min(P - u, R)$. The per-agent trade partition is exact. The ownership side-condition is respected. The eager and jitted runs agree. The full battery runs. I8 capacity headroom holds (five ceilings), and so does I9 non-vacuity. Every quote matches its source byte for byte.

**Failure implicates:** a leak in the participant view. Counterparty identity or a foreign hidden remainder would reach what a participant reads, which would invalidate any result obtained through the `Session` by letting information pass for skill.

**The two agent-facing surfaces under test.**

1. **The participant view**, i.e. everything the `Session` hands the participant:
   - after `reset` and after each step: `Session.top_of_book(state)` (`has_bid`, `has_ask`, `p_bid`, `p_ask`, `mid`, `q_bid`, `q_ask`) and the anonymous depth view `get_anonymous_lob(state.book)` (price and visible quantity per level, both sides);
   - after each step: the `FillReport` (`submitted`, `order_id`, `submit`, `passive_qty_signed`, `passive_value_eur`, `inventory_delta`, `cash_delta_eur`, `trade_rows`, `trades_saturated`) and the `done` flag;
   - at the end of the episode: the participant's own trades `get_agent_trades(final_book, AGENT_TRADER_ID)`, its own resting orders `get_agent_orders(final_book, AGENT_TRADER_ID)`, and its own accounting `state.agent_cash` and `state.agent_inventory`.

   The participant owns no row on the tape, so **every** tape participant is foreign to it. The returned `state` itself is the simulation carry, not a participant reader (finding F22).
2. **The per-agent trade report** `get_agent_trades(a)`, for every participant $a$ on the tape (and for two ids that never trade).

**The construction** (built in section 4 from a seeded PCG64 stream; there is no generator registry key for C13). One contract, one fresh book, $R$ rows, then the twins:

| phase | rows | what is drawn |
|---|---|---|
| participants | — | 3 or 4 distinct agent ids drawn from 1–999 (never the `Session` participant's id), plus one more id that places no order |
| resting book | 4–10 rows, NON or **ICE** (iceberg, probability 0.4, at least one), bids strictly below the mid, asks strictly above, so nothing crosses on entry | each row's owner drawn among the participants |
| order flow | 3–10 rows, NON or IOC (0.5 each), price anywhere in mid ± half-width, 0.1–2 MW | owners drawn among the participants |
| iceberg sizes | every iceberg's **visible peak exceeds every quantity that could ever trade against it** (all opposite-side flow, plus the participant's largest order, `Q_AGENT_MAX` ticks, on every step), and $\lceil \mathit{total}/\mathit{peak}\rceil \in [2, 8]$ | so no slice is ever exhausted and no hidden quantity can legitimately surface (finding F4) |
| order ids | distinct random ids in 1–99 999 | so no check can lean on ids being 1..N |
| participant | one random `AgentOrder` sequence, one entry per step: no order (`AgentOrder.none()`) with probability 0.3, otherwise a random order of side, price, quantity, execution restriction and peak drawn as in section 4 | the `random` policy; the `noop` policy submits `AgentOrder.none()` on every step |

**The twins**, each the same rows in the same order with the same prices, quantities and timestamps, except:

| world | foreign icebergs | agent ids | order ids | used for |
|---|---|---|---|---|
| `A` | ICE rows, hidden remainder resting | as drawn | as drawn | the reference world |
| `B` | each becomes a **NON order of its peak** (hidden remainder removed) | every participant relabelled by a derangement | every order id relabelled by a derangement | the participant-view twin (every tape participant is foreign to the `Session` participant) |
| `B@a`, one per participant $a$ | as in `B`, **except $a$'s own icebergs, kept with their hidden remainder** | $a$ kept; every other participant relabelled by a derangement | $a$'s kept; every other order id relabelled by a derangement | the trade-report twin of observer $a$ (the ownership side-condition) |
| `CTRL` | as in `B`, plus **one tick** added to the visible quantity of one iceberg that is at the touch | as in `B` | as in `B` | the positive control: `top_of_book` must see a one-tick change of the visible touch |

**What is asserted, per valid draw** (expected and observed recorded **in full** as canonical compact JSON; the participant view as one blake2b digest per step of `top_of_book`, `get_anonymous_lob`, `FillReport` and `done`, plus every differing step and field in full when they differ):

1. **Engine, both worlds.** Every trade row `[buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission]` of `A` and of `B` is pinned by the reference walk (D3-08); `B`'s rows are `A`'s relabelled; the per-order fates (filled, still resting, hidden remaining, IOC remainders cancelled, nothing evicted).
2. **Participant-view twin**, for the `noop` and the `random` policy, **where the premise holds** (*no tape iceberg fills a resting participant order on entry*, finding F17): `A` and `B` give a bit-identical participant view on every step: `top_of_book`, `get_anonymous_lob`, `FillReport` and `done`. The participant's own trades, own orders, cash and inventory at the end are bit-identical too. **Where the premise fails** (only the `random` policy rests participant orders), the **rule-implied outcome** of the amended item (4) is asserted instead, as (2r). The iceberg swept $M_j = f_j$ ticks, all of them the participant's resting quantity (its own icebergs' hidden part included), with $1 \le f_j < P_j$. So its remainder $R_j = T_j - f_j$ rests as instance $k_e = \lfloor M_j/P_j \rfloor = 0$ at its own price $b_j$, showing $v_1 = \min(P_j - u_j, R_j) = P_j - f_j$ ($u_j = f_j$) and hiding $R_j - v_1 = h_j$. `B`'s NON row rests the residual $P_j - f_j$ too. The rule-implied difference of the visible book is therefore **zero**: `A`'s visible quantity at that level equals `B`'s, `A`'s hidden remainder is still $h_j$, and `A`'s row rests at $b_j$. The participant view is bit-identical on **every** step, as where the premise holds.
3. **Positive control:** `CTRL` and `B` give a `top_of_book` that differs on at least one step (`noop`); `get_anonymous_lob` differs on that step as well.
4. **Trade-report partition:** for every participant, the absent id and the participant's id (`AGENT_TRADER_ID`, which has an empty report in the engine worlds: they carry no `Session` participant), the report holds exactly that id's legs `[price_int, qty_int]` (no side: correction J4, finding F16); the union over all ids is every trade's leg exactly twice, once from its buyer's report and once from its seller's.
5. **Trade-report twin:** for every participant $a$, $a$'s report in `A` and in `B@a` is bit-identical.
6. **Ownership side-condition:** each iceberg rests under its owner's identity with its full remaining quantity held (visible + hidden = total − filled); the twins removed only foreign remainders.

   6″. **Ownership side-condition, participant side** (§9.2 "Own orders", finding F6): in every `random` episode, `get_agent_orders(final_book, AGENT_TRADER_ID)` lists exactly the participant's resting orders, each participant iceberg at its full remaining quantity (visible + hidden), and it is bit-identical between `A` and `B`. `top_of_book` and `get_anonymous_lob` show the same participant iceberg only by its visible slice (finding F23).
7. **The twin stayed a twin:** in every `A` episode each foreign iceberg's hidden remainder is intact at the end (no slice regenerated), an iceberg of (2r) included, whose rule-implied hidden remainder $R_j - v_1$ is again $h_j$ (K2 r4); `B`/`CTRL` hold no hidden quantity of a tape order. The participant's own icebergs are excluded from "no slice regenerated": they may legitimately regenerate, identically in both worlds (finding F4).

Then the battery on `A` and `B`, the dataset precondition, the discrimination and coverage checks, the capacity checks and the eager comparisons (section 7).

**Re-run (`OMIE-CERT-RERUN-1`, K2 r2).** One correction is applied and nothing else in the expectation changes. **J4 — the trade-report legs carry no side:** the expected legs are `[price_int, qty_int]` (section 5 (4)); the first run demanded `[price_int, qty_int, side]`, a field §9.2's "Trade book" row does not list. The own-orders assertion (6′) is unchanged; `OMIE-OWN-ORDERS-ICE-QTY-1` (the first run's defect: an iceberg's owner was shown only the visible peak) has merged, so (6′) is expected to pass (S6). Carry-forward judging rules, binding: a battery `not_applicable` counts as vacuously satisfied, never as "not pass" (F9); every list is recorded in full, never cut; the eager runs are the first valid seeds (section 3).

**Re-run (`OMIE-CERT-RERUN-2`, K2 r3; verdict `SCOPE-CHANGE`, reading (a)).** The r2 run failed on four `random`-policy draws, where a foreign tape iceberg, entering the book, swept the RL agent's own resting order and then showed $\min(P, R)$ in `A` against $P - f$ in `B`. The verdict: this is surfacing **implied by the rules** (§4.2.1 Rule 2, *Incoming iceberg* (1) and (4); OMIE Rules Annex §4.4, `:10256-10261`), not a leak, because anonymity concerns **identity** (§7.1, §7.3). One correction is applied: a **premise check**, *no tape iceberg fills a resting agent order on entry*, and where it fails the **rule-implied difference** is asserted instead of twin equality (section 5 (2r), finding F17). The fallback the correction allows (discard those draws, counted and reported) is **not taken**: the rule-implied assertion is exact, in integer ticks, on each episode's book (F17). The premise-failed draws are still counted and listed in full (r2 saw 4 of 500, 0.8 %). Nothing else in the expectation changes; the r2 rules carry forward (J4; `not_applicable` vacuously satisfied, F9; every list in full; eager on the first valid seeds). Finding F16 is closed as the verdict directs: **C13 does not certify the trade side.**

**Re-run (`OMIE-CERT-RERUN-3`, K2 r4; market-spec v2.14 §4.2.1 Rule 2 *Incoming iceberg* items (2), (4), (6) amended 2026-10-20, `OMIE-SPEC-ICE-INCOMING-AMEND-1`, user ruling 2026-10-20).** The correction, binding: *"C13 — K2. Condition 'E-B or a changed item (4)': both hold. Re-derive the premise and the rule-implied difference from the amended item (4); the positive control is kept non-vacuous under C3."* Amended item (4) rests the remainder of an incoming iceberg as instance $k_e = \lfloor M/P \rfloor$ at $b + k_e\,d$, showing $v_1 = \min(P - u, R)$ with $u = M - k_e P$ (display C3 of the OQ-34 resolution of 2026-10-20); r3 used $\min(P, R)$ (display C1, rejected by the data). **Re-derived (F17):** the premise and its check are unchanged (a trade whose aggressor is a tape iceberg), but on this construction every entry sweep is $M_j = f_j \le$ `AGENT_QTY_MAX` $\times K < P_j$, so $k_e = 0$, $u_j = f_j \ge 1$ and $v_1 = P_j - f_j$, exactly `B`'s NON residual: the rule-implied difference is **zero**. (2r) now asserts the book identities ($\mathrm{vis}_A - \mathrm{vis}_B = 0$, $\mathrm{hid}_A = h_j$, one resting row at $b_j$) **and** bit-identity of the observation, the reward, the done flags and the report on every step (r3 asserted the observation and reward only before the entry step; this also covers the K4 r3 low note on post-entry hashes). Every premise-failed draw has $u > 0$, where C3 and the retired C1 disagree: a C1 display would put `A` $f_j$ ticks above `B` and fail (2r). **Positive control (F19):** (3) is unchanged and is exercised on every valid draw; a new aggregate assertion requires at least one valid draw on which the C3 display is exercised (premise failed, $u > 0$). The retired market-spec quotes of the old item (4) are replaced by the amended text (section 2). Nothing else in the expectation changes; the r2/r3 rules carry forward (J4; `not_applicable` vacuously satisfied, F9; every list in full; eager on the first valid seeds; C13 does not certify the trade side, F16).

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.14** (v2.13 touched §4.2.1 Rule 3 and §6.2 item 7, OQ-31, and no regeneration happens in C13; v2.14 added the §9.2 trade-side addendum, finding F16). Every quote is checked against the current file at run time. *(K2 r4)* §4.2.1 Rule 2 *Incoming iceberg* items (2), (4), (6) were rewritten on 2026-10-20 under the kept v2.14 token, and that amendment is **not** additive (market-spec version row v2.14): the two r3 quotes of the old item (4) no longer exist in the source and are replaced by quotes of the amended text.
- **F2 — the dataset is built in the notebook.** The generator has no C13 mode. Section 4 builds every tape with the generator's public helpers (`TapeBuilder`, `tape_capacity_bounds`, `check_envelope`, `Draw`) from `numpy.random.Generator(PCG64(seed))`; the twins travel as `Draw.aux_tapes`. A seed whose analytic bounds breach the envelope (`CapacityBreach`), or none of whose 64 attempts passes the acceptance test of section 4, is refused and counted as a discard. The results JSON lists no generator digest (`constructions=()`); section 4 prints the construction's digest over seeds 0–7 with the `construction_digest` formula. `draw.expected` is empty and never read.
- **F3 — the participant view is compared, never interpreted.** The readers' fields are named (section 1), but no expectation depends on their encoding, on how prices and quantities are scaled into them, or on what a field means: the twin is compared bit for bit, so any encoding passes and any dependence on a foreign hidden remainder or a foreign identifier fails. The `FillReport` and the done flag are compared too: they are the participant's per-step knowledge of its own fills, cash and inventory deltas. Two runs of the same mode are compared, so float bits are deterministic; eager against jitted, float fields are reported and not asserted (they legitimately differ between modes, V2/V10), and the integer state is asserted.
- **F4 — what the twin can and cannot certify.** §4.2: when a slice is exhausted a new one is activated, so hidden quantity *legitimately* becomes visible; §4.2.1 Rule 1: a FOK's pre-check *legitimately* reacts to hidden quantity. After either event the two worlds may differ without any leak. The construction therefore draws **no tape FOK**, places tape icebergs only as non-crossing resting orders, and sizes every peak above the total quantity that could ever trade against it (all opposite-side flow plus the participant's largest possible order, `Q_AGENT_MAX` ticks, on every step), so no foreign slice is exhausted in either policy; section 7 asserts from the run that every hidden remainder is still intact at the end of every `A` episode. C13 certifies that a **resting** hidden remainder and foreign identifiers cannot reach the participant; what hidden quantity does once it legitimately surfaces is C08–C10's. *(REL-09)* The participant **does** place FOK orders; by the peak bound no participant FOK decision can depend on a foreign hidden remainder (finding F25). The participant's own icebergs may regenerate (tape flow can exhaust their small slices); that is legitimate, the same in both worlds, and outside (7). *(K2 r3)* The peak bound covers every quantity that can trade **against** a resting iceberg. It does not cover the iceberg's own **entry**: a book-phase iceberg row never crosses the tape's own book (section 4), but it can cross a resting order the participant placed in an earlier step, and then §4.2.1 Rule 2 (*Incoming iceberg*) sweeps that order with the iceberg's total quantity and rests the remainder as an iceberg (since K2 r4, amended item (4): as instance $k_e = \lfloor M/P \rfloor$, showing $v_1 = \min(P - u, R)$). That is legitimate surfacing, not a leak. The construction does not exclude it (the participant's random prices span the order-flow band, which straddles the resting book); it is handled by the premise check of F17.
- **F5 — "hidden remainder removed" is a NON order of the peak.** The specification does not say whether an iceberg with total = peak is a valid order; a plain NON order at the same price and position, of the peak's quantity, is exactly "the book as shown" (§4.2 "Only the peak size is shown in the order book"), and behaves identically while its peak is not exhausted (§4.5 ICE "behave like NON orders"). That is the twin used.
- **F6 — the ownership side-condition is certified at the engine, in the twin structure and on the participant's own-orders reader.** *(REL-09)* The release's participant can own icebergs (`AgentOrder` with ICE and a peak), and the specification's "Own orders" row has its counterpart in `get_agent_orders(final_book, AGENT_TRADER_ID)`; assertion (6″) asserts the positive half there, for the participant's own icebergs. For the tape owners the positive half ("own-iceberg full quantity **is** visible to its owner") is asserted as: each iceberg rests under its owner's identity with visible + hidden = total − filled (the identity the engine must keep, §7.3); and the negative half is side-conditioned exactly as the review requires: the observer twin `B@a` keeps $a$'s own icebergs and removes only foreign remainders. Section 6 exposes the owner-facing view of own orders as `own_orders_view` (the first run's verdict names `ContractBook.get_agent_orders`), and section 7 then asserts (6′): it shows each owner exactly its own resting orders, each iceberg at its full remaining quantity; after `OMIE-OWN-ORDERS-ICE-QTY-1` this is expected to pass (S6). If no such view is exposed, that is **recorded, not asserted** (a §9.2 representation gap, not a leak).
- **F7 — "no counterparty identifier" is asserted metamorphically, per observer.** In `B@a` every identifier other than $a$'s own is relabelled; $a$'s own report must be byte-identical. A report may name its owner (§7.1: *"identifying solely those operations performed by the agent itself"*); it may not name, or depend on, a counterparty. Reading only the report, this needs no knowledge of its row layout. *Risk noted, not resolved here:* review V3-18 records that the `(M, 6)` settlement row carries `buyer_agent_id` and `seller_agent_id`; if `get_agent_trades` returns such rows unredacted, this assertion fails for every observer who traded — which is the reading of §7.2 the card asks for.
- **F8 — no self-trade is drawn.** §7.2 does not say whether a trade whose buyer and seller are the same agent is notified once or twice; an attempt whose walk contains one is redrawn. (Self-matching itself is C03's and M6's.) This holds for the tape. The participant's self-matches are allowed (OMIE has no self-trade prevention, §6.6); they are identical in both worlds and reported, not asserted.
- **F9 — which battery checks must pass, and which may be not applicable.** Every valid draw has at least one trade. On both engine worlds no check of RB and I1–I7, I9 may fail: RB, I2, I3, I6, I7 and I9 are expected to **pass** and I1, I4 and I5 may be `not_applicable`; under the carry-forward rule a `not_applicable` status counts as **vacuously satisfied** for every one of them, never as "not pass" (counts reported). I8 `invalid` on either world discards the draw.
- **F10 — a capacity flag inside the margin is a failure, not only a discard** (as C08–C12), including the env runs, whose analytic margin adds the participant's orders: at most one per step, of at most `Q_AGENT_MAX` ticks, so at most `Q_AGENT_MAX` trades, one resting row and one iceberg slot per step.
- **F11 — the env configuration is part of the expectation.** `ENV_CONFIG` (section 3) fixes four tape rows per step; *(REL-09)* the quantity buckets no longer exist, and the participant's order size is bounded by the draw distribution instead (`Q_AGENT_MAX` = 6 ticks, section 4, step 5), because the peak bound of F4 depends on it; section 6 must build `Session` with exactly that configuration and the tapes, nothing else.
- **F12 — identifiers are varied, not held constant.** Participant ids, the absent id and order ids are drawn at random on every seed, so no check can pass only because an id happens to be small, ordered or equal to a row index.
- **F13 — the session's internal per-agent trade tape** (`state.agent_trades`) is reported, not asserted: §7.3 lets the engine keep identity internally. The `FillReport.trade_rows` that the participant receives are asserted (they are part of the participant view); they carry a constant anonymous counterparty id.
- **F14 — not drawn:** a tape FOK, tape icebergs that cross the tape's own book on entry (an entering iceberg can still cross the participant's resting order: F4, F17), ICEwPI (participant or tape), GTD, slice regeneration, ICEwPI, cancellations, gate expiry, more than one contract, self-trades. The envelope (mid 5–300 EUR/MWh, half-width 2–20 ticks, row counts, quantities, participant count) is an empirical test range, never a bound; every price and quantity stays inside §2.3.
- **F15 — signature seen.** The method name and signature `ContractBook.get_agent_trades(state, agent_id) -> list` were seen in a repository-wide search for the card's name; the body and docstring of `contract_book.py` were not read (independence rule 1). Section 6 decides how to obtain the book state. The name `ContractBook.get_agent_orders` was read only in the first run's verdict, as quoted in the K2 r2 brief.
- **F16 — the trade side (§9.2, v2.14): the correction and the specification now disagree.** As instructed (J4), the expected legs carry no side. But market-spec v2.14 (`OMIE-SPEC-CERT-AMEND-1`) adds, after the §9.2 table: *"Besides price, quantity, timestamp and contract, each own trade in the trade book also carries its **side**: BUY or SELL, from the agent's point of view, which is the side of the agent's own order in that trade."* (quoted in section 2). Read literally, side is now a required field of the trade book; the same addendum says *spec only, no engine change filed, by user decision*. The expectation does **not** reintroduce it. *(K2 r3, closed by the `OMIE-CERT-RERUN-2` verdict.)* **C13 does not certify the trade side:** the gap is carried as a row of the certificate's §9.2 gap table (`OMIE-CERT-LADDER-DOCS-1`), never as a C13 assertion.
- **F17 — the premise check and the rule-implied display (K2 r3; re-derived at K2 r4 from the amended item (4)).** For every `A`/`B` episode pair, section 7 reads the episode's final book (`run_env`'s `book_final`, the harness's `BookSnapshot`) and lists every trade whose **aggressor** is a tape iceberg $j$ of `A` (in `B`: the NON row $\tau(j)$ that replaces it). Here an iceberg is an aggressor only on entry (no slice is ever regenerated, F4), so these are exactly its entry fills. **The premise holds** when there is no such trade in either world: assertion (2), the bit-identical twin. **The premise fails** otherwise: assertion (2r). Every entry trade of $j$ must have the participant as its passive side (the tape's own book never crosses an entering iceberg, by (W)), so item (4)'s swept quantity is $M_j = f_j$, the participant's total fill on that entry (its own icebergs' hidden part included: Rule 2 multi-slice sweeps their slices in turn), and $f_j$ must be the same for $\tau(j)$ in `B`. The participant's resting quantity is at most `Q_AGENT_MAX` $\times K$, below $P_j$ by the peak bound (section 4, step 2; checked by the precondition), so $1 \le f_j < P_j$ and both worlds sweep all of it. Item (6) with $d = 0$: every instance has limit $b_j$, so the sweep never stops at an instance boundary; and as $f_j < P_j$ the whole sweep lies in instance 0, so item (2)'s split at an incoming instance boundary never applies (one row per resting agent entry, in both worlds). Then, by item (4): $k_e = \lfloor f_j/P_j \rfloor = 0$, $u_j = f_j - 0 \cdot P_j = f_j$, and $R_j = T_j - f_j \ge P_j + 1 - f_j$ (as $T_j > P_j$); `A` rests $j$ as instance 0 at its limit $b_j + 0 \cdot d = b_j$, showing $v_1 = \min(P_j - u_j, R_j) = P_j - f_j$ and hiding $R_j - v_1 = T_j - P_j = h_j$, with no new timestamp (§5.2); `B` rests the NON residual $P_j - f_j$ of $\tau(j)$ at its entry timestamp (§6.2 item 4). So the two visible books are equal in price, quantity and time priority at every instant; later fills $g_j$ are the same in both worlds and exhaust neither ($f_j + g_j \le$ the peak bound $< P_j$). At the end of the episode
  $\mathrm{vis}_A(j) - \mathrm{vis}_B(\tau(j)) = v_1 - (P_j - f_j) = 0$ and $\mathrm{hid}_A(j) = R_j - v_1 = h_j$ (section 5 (2r)). Nothing the participant may see differs, so the participant view (`top_of_book`, `get_anonymous_lob`, `FillReport`, done flags, own trades, own orders, cash, inventory) is asserted **bit-identical on every step**. (r3, under the retired display $\min(P, R)$, had `A` show $\min(f_j, h_j)$ more than `B` from the entry step $k^*$ on, and asserted the observation and reward only before $k^*$.) F3 still holds: the participant view is compared, never interpreted, and the book identities are asserted on the book the env holds. Every premise-failed draw is listed in section 7 with its seed, policy, icebergs, $f_j$ and $u_j$. The fallback (discarding those draws) stays untaken.
- **F18 — what the premise check relies on.** It reads the episode's trade rows by aggressor order id (the harness reader `_fill_from_row` of `omie_invariants`), and it needs the episode's trade rows to persist to the end of the episode (the same rows `own_report` is read from). If the engine recorded an entering iceberg's fills under another aggressor id, or dropped them before the end of the episode, the premise would look intact while the twin differs, and (2) would fail loudly, never silently. That would be escalated, not reinterpreted.

- **F19 — the positive control under display C3 (K2 r4).** Two things, kept apart. **(a) The positive control (3)**, `CTRL` against `B` on the `noop` policy (one extra visible tick on an iceberg at the touch), involves no entry sweep (`noop` places no agent order) and is unchanged; section 4's acceptance test requires an iceberg at the touch on a noop step, so it is exercised on **every** valid draw. **(b) No draw can carry a non-zero rule-implied twin difference under C3.** A non-zero $v_1 - (P - f)$ needs $R < P - u$, i.e. $T < (k_e + 1)P$: the remainder ends inside instance $k_e$ with $k_e \ge 1$, so $M \ge P$. That contradicts the peak bound that makes the twin execution-equivalent (F4), and `B`'s NON row of the peak would then be exhausted too, so the twins would differ in trades, not only in display. Under r3's C1 every premise-failed draw carried a non-zero difference ($\min(f_j, h_j) \ge 1$); under C3 the premise-failed draws are the **discriminating** case instead: $u_j = f_j > 0$ on every one of them, where C3 shows $P_j - f_j$ and the retired C1 shows $P_j$, so an engine still displaying C1 fails (2r) by $f_j$ ticks on the book and, very likely, on `top_of_book` and `get_anonymous_lob`. Section 7 asserts that at least one valid draw exercises it (premise failed, $u > 0$, jitted `random` episode); it is a property of the participant's random orders against the drawn prices, so it is counted from the run, not predicted. *Proposed, not added (generator or construction change, for the human):* if a C13 draw with a **non-zero** rule-implied difference is wanted, the construction needs a book-phase iceberg that crosses the tape's own resting liquidity on entry with $M \ge P$ and $T - M < P - u$ (so `A` shows $R$, not $P - u$); (W) refuses that today ("iceberg would cross on entry"), and its twin `B` would then trade differently, so its natural oracle is the engine-level display of C08's incoming-remainder class, not C13's twin.
- **F20 — the card's Known result row is not amended.** The card `OMIE-CERT-C13-1` still states r3's formula ($\min(P, R) - (P - f) = \min(f, h)$, hidden $\max(0, h - f)$) in its Known result row, and its Input artifacts row names market-spec v2.12. `scenario-ladder.md` §C13, the canonical spec, carries the 2026-10-20 amendment ($v_1 = \min(P - u, R)$). This expectation follows the ladder and the amended market-spec; aligning the card row is for the architect.
- **F21 — what of the amended rule C13 does not exercise.** ICEwPI ($d \ne 0$), the stop at the first instance that cannot trade (item (6)), $k_e \ge 1$ (a rest price $b + k_e\,d$ other than $b$) and the row split at an incoming instance boundary (item (2)) never occur here: no ICEwPI is drawn (F14) and every entry sweep is below one peak. They are C08's. C13 exercises item (4) with $k_e = 0$ and $u > 0$ only.
- **F22 — the carry is not a participant reader.** `Session.step` and `step_env` return the full `OMIEContractState`. Its book holds order ids and trader ids, and its iceberg table holds hidden remainders. The specification requires identity to be kept internally (§7.3 *"Maintain agent identity internally (for PnL accounting and observation)."*), so the carry is internal state, the same state the overseer's view inspects. C13 certifies the participant readers (`top_of_book`, `get_anonymous_lob`, `FillReport`, `done`, and `get_agent_trades` / `get_agent_orders` for the participant's own id). It does not certify the carry. A user who reads `state.book` directly is inspecting the market operator's state. This is a scope statement, not a gap.
- **F23 — a participant's own iceberg in the anonymous view.** `top_of_book` and `get_anonymous_lob` show the participant's own iceberg only by its visible slice, as they show any iceberg (§9.2 "LOB per contract": *aggregated visible quantities*). Its full quantity is on the own-orders reader (§9.2 "Own orders"). Two rows, two readers: this is not a violation of the ownership side-condition, which grants a permission on the own-orders reader, not on the public book.
- **F24 — LOB timestamps.** §9.2's "LOB per contract" row lists timestamps. Neither `top_of_book` nor `get_anonymous_lob` returns any. This is a representation gap, not a leak. It is recorded, not asserted.
- **F25 — why a participant FOK cannot react to a foreign hidden remainder here.** A participant FOK of $q \le$ `Q_AGENT_MAX` ticks is accepted iff the crossing matchable quantity is at least $q$. If a foreign iceberg crosses, its visible quantity at that moment is $P_j$ minus what has already traded against it. That is still greater than $q$, because the peak bound exceeds all opposite-side flow plus `Q_AGENT_MAX` $\times K$. So the FOK is accepted in both worlds. If no foreign iceberg crosses, the two worlds' crossing liquidity is the same. Either way the decision is the same in `A` and `B`. §4.2.1 Rule 1 still holds; it cannot discriminate on this construction. The participant's own icebergs are identical in both worlds.

## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **What an agent may see (§7.1, §7.3, §9.2 "LOB per contract" and "Best bid / best ask"; OQ-07; review §2).** The book is shown anonymously, as prices and aggregated *visible* quantities, identifying only the agent's own operations. Hence the participant-view twin: a foreign hidden remainder and foreign identifiers must not change one bit of what the participant reads through `top_of_book`, `get_anonymous_lob`, its fill reports or its own-trade and own-order readers.
- **What an agent is told of trades (§7.2, §9.2 "Trade book").** Each agent is notified of its own trades only, without the counterparty. Hence the partition, the union and the per-observer report twin. The legs compared are `[price_int, qty_int]` (correction J4); the v2.14 side addendum is quoted for the record and **not asserted** (finding F16).
- **The ownership side-condition (§9.2 "Own orders").** An owner may see its own full iceberg quantity, so only *foreign* remainders are removed, and the owner's full quantity must be held.
- **When hidden quantity legitimately surfaces (§4.2; §4.2.1 Rule 1; §4.5).** A regenerated slice and a FOK pre-check legitimately depend on hidden quantity; the construction excludes both, so the twins stay execution-equivalent.
- **When a foreign iceberg meets the participant on entry (§4.2.1 Rule 2, *Incoming iceberg* (1)–(4), (6), amended 2026-10-20; OQ-34 resolution, display C3; Annex §4.4; K2 r3, re-derived at K2 r4).** An entering iceberg that crosses aggresses with its total quantity, as a chain of instances of size $P$ (all at limit $b$ when $d = 0$), and rests its remainder $R$ as instance $k_e = \lfloor M/P \rfloor$ at $b + k_e\,d$, showing $v_1 = \min(P - u, R)$, $u = M - k_e P$, with $R - v_1$ hidden and no new timestamp. When it sweeps the participant's own resting order (its own icebergs' hidden part included) ($M = f < P$, so $k_e = 0$ and $u = f$) it shows $P - f$: exactly what `B`'s NON row of the peak rests. The rule-implied difference is zero, so (2r) asserts the book identities and the bit-identical twin on every step (findings F17, F19). Anonymity concerns identity (§7.1, §7.3).
- **What the participant may submit (§9.1 "Submit order"; §2.3; §4.2; §4.5; §4.6).** The participant's random orders use the actions and bounds of the specification: BUY or SELL, a price on the 0.01 EUR/MWh tick, a quantity on the 0.1 MW tick, NON, IOC, FOK or ICE (icebergs carry a peak), GFS validity. Its orders never exceed `Q_AGENT_MAX` ticks, below every foreign peak. So none of them can exhaust a foreign peak or make a FOK pre-check depend on a foreign hidden remainder (findings F4, F25).
- **Why the twins execute identically (§6.6, §5.2).** Matching ignores ownership, and priority is price then timestamp, so relabelling agents and order ids changes no trade.
- **The reference walk (§6.1; §6.2 items 2–6; §4.5; review D3-08).** It pins every trade row of both engine worlds, counterparties included.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §7: anonymity and information visibility ----------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§7.1',
        'OMIE Rules §45.2.12 explicitly states: "information available on the Market Operator\'s Market Platform for market agents regarding products, contracts and data on the orders contained on the list of local orders is to be displayed on an **anonymous basis**, identifying solely those operations performed by the agent itself."',
        note='The anonymous book: what an agent may see of the book identifies only its own operations. Asserted as the participant-view twin (a foreign hidden remainder and foreign identifiers cannot change one bit of the participant view).',
    ),
    SpecQuote(
        'market-spec.md',
        '§7.2',
        'Each agent sees only their own matched trades.',
        note="The trade report of agent a holds exactly a's fills: the per-agent partition.",
    ),
    SpecQuote(
        'market-spec.md',
        '§7.2',
        'OMIE Rules §46.1: "the result is to be notified to the agent in question."',
        note='Every trade is notified to its buyer and to its seller, and to no one else: the union over agents reconstructs the trade tape, each trade once per side.',
    ),
    SpecQuote(
        'market-spec.md',
        '§7.2',
        'Counterparty identity is not disclosed.',
        note="Asserted as the report twin: permuting every identifier other than the observer's own leaves the observer's report byte-identical.",
    ),
    SpecQuote(
        'market-spec.md',
        '§7.3',
        'Maintain agent identity internally (for PnL accounting and observation).',
        note='Identity is kept inside the engine: the engine-side records (trade rows, owners of resting orders) carry agent ids and are pinned exactly; only the agent-facing surfaces must be blind to foreign ones.',
    ),
    SpecQuote(
        'market-spec.md',
        '§7.3',
        'Expose the LOB to agents as anonymous (prices and quantities only, no counterparty identity).',
        note='The participant view (top_of_book, get_anonymous_lob) may depend on visible prices and quantities only.',
    ),
    SpecQuote(
        'market-spec.md',
        '§7.3',
        "Each agent's observation includes only their own pending orders and matched trades.",
        note="The participant's own trades and own orders are part of the participant-view twin; other agents' trades never reach it.",
    ),
    # --- §9.2: the observable variables, and the ownership side-condition ---------------------------
    SpecQuote(
        'market-spec.md',
        '§9.2',
        '| LOB per contract | Bid/ask price levels with aggregated visible quantities and timestamps. **Anonymous** — counterparty identity not shown (OMIE Rules §45.2.12). |',
        note='Aggregated VISIBLE quantities: a hidden remainder is not part of the book an agent sees.',
    ),
    SpecQuote(
        'market-spec.md',
        '§9.2',
        "| Own orders | Agent's own active orders (showing full iceberg quantity) |",
        note="The ownership side-condition: an owner may see its own full iceberg quantity, so the twin removes FOREIGN hidden remainders only, never the observer's own; asserted on the participant's own-orders reader as (6″).",
    ),
    SpecQuote(
        'market-spec.md',
        '§9.2',
        '| Best bid / best ask | Top-of-book per contract |',
        note="`top_of_book` is the specification's \"Best bid / best ask\" observable. It is held to the same anonymity as the LOB row: prices and visible quantities only.",
    ),
    SpecQuote(
        'market-spec.md',
        '§9.2',
        '| Trade book | Executed trades: price, quantity, timestamp, contract. Own trades only (OMIE Rules §46.1). |',
        note='The trade book row: price, quantity, timestamp, contract; own trades only. The legs compared are [price_int, qty_int] (correction J4).',
    ),
    SpecQuote(
        'market-spec.md',
        '§9.2',
        "Besides price, quantity, timestamp and contract, each own trade in the trade book also carries its **side**: BUY or SELL, from the agent's point of view, which is the side of the agent's own order in that trade.",
        note='The v2.14 trade-side addendum. Recorded, not asserted: the J4 correction drops the side from the expected legs (finding F16).',
    ),
    # --- icebergs: what is hidden, and when it legitimately stops being hidden ----------------------
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'A regular limit order with hidden total quantity and visible peak quantity.',
        note='The hidden remainder is total - peak.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'Only the peak size is shown in the order book.',
        note='Why the twin may replace a foreign iceberg by a plain order of its peak: that is the book as shown.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'When a slice is exhausted, a new slice of peak size is activated and receives a new timestamp.',
        note='After a slice is exhausted, hidden quantity legitimately becomes visible: the twin is execution-equivalent only while no foreign peak is exhausted (finding F4), which the construction guarantees.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "A Fill-or-Kill order's pre-match liquidity check considers the total iceberg quantity for opposite-side icebergs at crossing prices.",
        note='A FOK legitimately reacts to hidden quantity, so no tape FOK is drawn (finding F4); a participant FOK cannot react to a foreign hidden remainder here (finding F25).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r"Below, $b$ is the incoming iceberg's price, $P$ its peak (reduced part), $d$ its price increment (PPD; $d = 0$ for a standard iceberg), $M$ the quantity it sweeps on entry and $R$ its remainder.",
        note="K2 r4: the notation of the amended Incoming-iceberg rule (2026-10-20). In C13, d = 0 (no ICEwPI is drawn) and M is the participant's resting quantity swept on the entry, M = f.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**(1) Total quantity.** It aggresses with its **total** quantity, not only its visible peak. It keeps matching while it remains executable and has quantity left.',
        note="K2 r3: an entering iceberg that crosses the participant's resting order sweeps it with its total quantity in A; in B the NON row of the peak sweeps the same quantity (the participant's resting quantity is below the peak). The premise check of finding F17.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r'**(2) One transaction per (counter bid, instance) pair.** The incoming iceberg is a chain of instances, each of size $P$; the last instance may be smaller.',
        note="K2 r4: the incoming iceberg as a chain of instances. In C13 the sweep f < P lies inside instance 0, so no row splits at an incoming instance boundary (finding F17, F21).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r'A trade row ends where a resting entry ends **or** where one incoming instance ends and the next begins.',
        note="K2 r4: in C13 only the first case occurs (one row per resting agent entry), in A as in B.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r'**(4) Remainder.** If a remainder $R > 0$ is left once the order is no longer executable, it enters the book **as an iceberg** with total $R$.',
        note="K2 r4, assertion (2r): the remainder R = T - f of a tape iceberg whose entry swept f ticks of the participant's resting quantity rests as an iceberg.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r'With $k_e = \lfloor M/P \rfloor$ and $u = M - k_e P$ (the part of instance $k_e$ already traded), the remainder keeps its place in the ladder: it continues as instance $k_e$, at limit $b + k_e\,d$.',
        note="K2 r4: M = f < P, so k_e = 0 and u = f; the remainder rests as instance 0 at the iceberg's own price b (asserted: one resting row of the iceberg, at b).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r'It shows $v_1 = \min(P - u, R)$, the unused part of instance $k_e$, and keeps $R - v_1$ hidden.',
        note="K2 r4, assertion (2r): A shows v1 = min(P - f, T - f) = P - f, exactly B's NON residual, so vis_A - vis_B = v1 - (P - f) = 0; A's hidden remainder is R - v1 = T - P = h, still intact (findings F17, F19).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'As for a NON residual, partial execution assigns no new timestamp (§5.2).',
        note="The remainder keeps its entry timestamp, as B's NON residual does (§6.2 item 4), so time priority and later fills are the same in both worlds (finding F17).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        r'Instance $k$ ($k = 0, 1, 2, \dots$) trades only at prices its own limit $b + k\,d$ crosses; for $d = 0$ every instance has limit $b$.',
        note="K2 r4: with d = 0 the sweep never stops at an instance boundary while b crosses; it sweeps the participant's crossing resting quantity in full, in A as in B.",
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-34',
        r'**Display C3 (derived from data).** With $u = M - k_e P$, the rested instance shows $v_1 = \min(P - u, R)$, its unused part, and $R - v_1$ is hidden.',
        block='**Resolved by user ruling (2026-10-20',
        note="K2 r4: the display the correction names (C3). Every premise-failed C13 draw has u = f > 0, the case where C3 (P - f) and the retired C1 (P) disagree (finding F19).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| ICE | Yes (peak slices) | Yes (hidden qty rests) | Iceberg only |',
        note='The hidden quantity rests in the book: it is there, and it must not be seen.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| NON | Yes | Yes | Regular, Block |',
        note='NON: partial fills, the remainder enters the book.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| IOC | Yes | No (cancelled) | Regular |',
        note='IOC: partial fills, never enters the book.',
    ),
    # --- what the participant may submit (REL-09: the random AgentOrders) ------------------------------
    SpecQuote(
        'market-spec.md',
        '§9.1',
        '| Submit order | contract_id, side (BUY/SELL), quantity (MW), price (EUR/MWh), execution_restriction (NON/IOC/FOK), validity (GFS/GTD + expiry_time), order_type (regular/iceberg/block) | For iceberg: also peak_size [, price_delta] |',
        note="The participant's random orders are drawn from this action.",
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Price tick (minimum increment):** 0.01 EUR/MWh (2 decimal places maximum).',
        note='The random price is an integer number of 0.01 EUR/MWh ticks.',
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).',
        note='The random quantity and peak are at least 1 tick.',
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Quantity tick (minimum increment):** 0.1 MW (1 decimal place maximum).',
        note='The random quantity and peak are integer ticks.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'Icebergs behave like NON orders (resting, partial fills allowed) but carry hidden total quantity.',
        note='A participant iceberg rests like a NON order and carries a hidden part that only its owner sees.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'icebergs cannot combine with IOC or FOK behavior',
        note='ICE is its own restriction in the draw, never combined with IOC or FOK.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| FOK | No (full or cancel) | No (cancelled) | Regular, Linked basket |',
        note='A participant FOK never rests. Its pre-check is the one place hidden quantity can matter; finding F25 shows it cannot here.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.6',
        "| GFS | Valid until contract's GCT | Yes (default) |",
        note='Every random order is GFS. GTD is not drawn (F14).',
    ),
    # --- why permuting identities leaves execution unchanged -----------------------------------------
    SpecQuote(
        'market-spec.md',
        '§6.6',
        '**Simulator implication:** The matching engine must not reject or skip matches based on order ownership.',
        note='Matching is ownership-blind, so relabelling participants changes no trade: the twin worlds execute identically.',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.2',
        'All timestamps across orders and explicit capacity requests are assigned from a single global queue (unique, sequential).',
        note='Priority is by price, then timestamp: order ids play no part, so permuting them changes no trade either.',
    ),
    # --- the reference walk -------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Buy side:** ranked highest price first; ties broken by earliest timestamp.',
        note='A SELL aggressor meets the highest bid first; equal prices by entry.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Sell side:** ranked lowest price first; ties broken by earliest timestamp.',
        note='A BUY aggressor meets the lowest ask first; equal prices by entry.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 2',
        'Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.',
        note='The crossing condition of the walk.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 3',
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Each trade row carries the resting order's price.",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 4',
        'If incoming order is partially filled and has NON restriction: residual enters the book with its original entry timestamp (see §5.2).',
        note='A NON residual rests, keeping its entry timestamp.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 5',
        'If incoming order is partially filled and has IOC restriction: residual is cancelled.',
        note='An IOC residual is cancelled (cause ioc_remainder in the record).',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 6',
        'If matching exhausts one price level, continues to next best price level.',
        note='An aggressor walks level after level.',
    ),
    # --- open questions and review --------------------------------------------------------------------
    SpecQuote(
        'open-questions.md',
        'OQ-07',
        "Maintain agent identity internally (for PnL accounting and observation). Expose the LOB to agents as anonymous. Each agent's observation includes only their own pending orders and matched trades. This matches the real market rule exactly.",
        block='**Modeling choice for simulator:**',
        note='The resolved modelling choice C13 certifies.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§2',
        "the agent's observation must be bit-identical to one taken from a counterfactual book with the foreign hidden remainder removed and identifiers permuted. Side-conditioned on ownership, since §7.3/§9.2 permit an agent to see *its own* full iceberg quantity.",
        note='The metamorphic formulation and its side-condition, as the review added C13.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'D3-08',
        "Any scenario that draws two same-price icebergs against one\naggressor (C05's interloper family, C08's multi-slice draw, C09) should pin the exact counterparty\nsequence in its Known result.",
        note='Good practice adopted here: every trade row of both worlds is pinned by buyer and seller order id and agent.',
    ),
)
CITATIONS = (
    Citation(
        'OMIE Day-Ahead and Intraday Electricity Market Rules, Annex, Iceberg bids §4.4 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:10256-10261',
        'When entering an “Iceberg” bid with a competitive price, the quantity to be considered will be the total quantity of the “Iceberg” bid, recording a transaction for each counter bid that it matches with. Additionally, if the total quantity does not match, the viewable quantity that will appear in the Bid Book will be at most the reduced part specified when creating the bid, even if the matched quantity was not a multiple of the reduced part.',
    ),
    Citation(
        'OMIE Day-Ahead and Intraday Electricity Market Rules §45.2.12 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:5458-5463',
        'During trading on the continuous intraday market, the information available on the Market Operator’s Market Platform for market agents regarding products, contracts and data on the orders contained on the list of local orders is to be displayed on an anonymous basis, identifying solely those operations performed by the agent itself.',
    ),
    Citation(
        'OMIE Day-Ahead and Intraday Electricity Market Rules §46.1 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:5494-5495',
        'When there is a match, its result is to be notified to the agent in question, with the list of local orders being updated.',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws (the card's default for a random scenario), from seed `SEED0`. A seed names one draw: world `A`, its twins and the random policy's `AgentOrder`s.
- **Discards.** A seed the section-4 builder refuses (`CapacityBreach`, or no accepted attempt), a draw whose tapes lie outside the capacity margin before the run (env-side bounds included, F10), or a draw the battery's I8 marks `invalid` on either engine world after the run, is **discarded and replaced by the next seed**. The loop stops at `DRAWS` valid draws or after `2 × DRAWS` attempts; the discard rate is reported. An I8 discard of a draw that passed the pre-run margin is also a failed assertion (finding F10).
- **The premise check is not a discard** (K2 r3, finding F17). A draw whose `random` episode has a tape iceberg sweeping the participant's resting order on entry stays valid and is judged by the rule-implied assertion (2r) instead of (2); section 7 counts those draws and lists them in full. *(K2 r4)* Under the amended item (4) the rule-implied difference is zero on this construction, so (2r) is the twin equality on every step plus the book identities (finding F17); at least one such draw must occur (finding F19).
- **`EAGER_DRAWS = 2`.** The first `EAGER_DRAWS` **valid** draws also run eager (the `A` engine run, the `A` reports and the four `A`/`B` episodes), compared with the jitted runs on every integer field; on top of these, an eager run is **forced** for the first valid draw of each eager class. A missing eager comparison for any eager class is a failed `eager_jit` assertion; eager runs are chosen among valid draws only.
- **`ENV_CONFIG`** fixes the session the participant-view twin runs in: `{"orders_per_step": 4}` plus the tapes (finding F11). The quantity buckets no longer exist; the participant's order size is bounded by the draw distribution instead (`Q_AGENT_MAX = 6` ticks).
- **The random `AgentOrder` distribution** (section 4, step 5): `P_NOOP`, `P_BUY`, `P_EXEC` and `Q_AGENT_MAX`, listed by the next cell; the price band is the order-flow band $[m - H, m + H]$ of the draw.
- **The classes** (each must appear among the valid draws: the coverage assertion of section 7) are listed by the next cell.

In [ ]:
CERT_ID = "C13"            # C00-C16 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                # valid draws (card: random scenarios default to 500 draws)
SEED0 = 0                  # first seed
EAGER_DRAWS = 2            # the first EAGER_DRAWS valid draws also run eager (plus the first of each eager class)

In [ ]:
import math
import pathlib
import types

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.encoding import encode_price, encode_qty
from omiecid_simulator.omiecid.omie_session import AGENT_TRADER_ID, AgentOrder
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

CONSTRUCTION = "C13.anonymity_twin (built in section 4)"   # not a generator registry key (finding F2)
CONFIG = OMIEProductConfig()          # builds tapes; never a source of the expectation
ENVELOPE = synth.DEFAULT_ENVELOPE
MAX_ATTEMPTS = 2 * DRAWS   # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts

BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)

# ---- the environment, as the expectation fixes it (finding F11) -------------------------------------
ORDERS_PER_STEP = 4                   # tape rows replayed per env step
ENV_CONFIG = {"orders_per_step": ORDERS_PER_STEP}   # + "tapes"; nothing else
PARTICIPANT_ID = int(AGENT_TRADER_ID) # the Session participant's identifier: it owns no tape row
POLICIES = ("noop", "random")
PREMISE = "no tape iceberg fills a resting participant order on entry"   # finding F17: (2) where it holds, (2r) where not

# ---- the random AgentOrders (section 4, step 5; REL-09) ----------------------------------------------
P_NOOP = 0.3                          # P(AgentOrder.none() on a step)
P_BUY = 0.5                           # P(an active order is BUY), else SELL
P_EXEC = {"NON": 0.40, "ICE": 0.30, "IOC": 0.15, "FOK": 0.15}   # execution restriction of an active order
EXEC_CUTS = ((0.40, NON), (0.70, ICE), (0.85, IOC), (1.0, FOK))   # cumulative P_EXEC, in draw order
Q_AGENT_MAX = 6                       # qty ticks (0.6 MW): the largest participant order, icebergs' total included

# ---- display units and the per-iceberg slice ceiling ------------------------------------------------
O_TICKS = 999_900                     # p_int of 0.00 EUR/MWh (§2.3 price floor -9,999 at 0.01): display, and the AgentOrder price
PRICE_TICK_EUR, QTY_TICK_MW = 0.01, 0.1
PRICE_INT_MAX, QTY_INT_MAX = 1_999_800, 9_990   # §2.3 bounds
MAX_SLICES = 64                       # capacity precondition: ceil(total / peak) <= 64

# ---- the draw envelope: empirical test ranges, never bounds (finding F14) ---------------------------
MID = (500, 30_000)                   # mid, in ticks above 0.00 EUR/MWh: 5-300 EUR/MWh
HALF_WIDTH = (2, 20)                  # price ticks either side of the mid
N_BOOK = (4, 10)                      # resting-book rows (NON or ICE), none crossing on entry
P_ICE = 0.4                           # P(a resting-book row is an iceberg); at least one is
N_FLOW = (3, 10)                      # order-flow rows (NON or IOC)
P_IOC = 0.5                           # P(an order-flow row is IOC)
Q_BOOK = (1, 50)                      # qty ticks of a resting NON row
Q_FLOW = (1, 20)                      # qty ticks of an order-flow row
PARTICIPANTS = (3, 4)                 # participants on the tape (>= 3: every observer has >= 2 others)
ID_POOL = (1, 999)                    # participant ids and the absent id
OID_POOL = (1, 99_999)                # order ids
SLICES = (2, 8)                       # ceil(total / peak) of each iceberg
PEAK_EXTRA = (1, 200)                 # ticks added to each iceberg's peak bound (finding F4)
MAX_TRIES = 64                        # attempts per seed before the seed is refused

# ---- the classes ------------------------------------------------------------------------------------
C_ICE_TRADED = "a foreign iceberg's visible peak traded (partially)"
C_TWO_OWNERS = "icebergs of two different owners (B@a keeps one, removes the other)"
C_ICE_BOTH = "icebergs on both sides of the book"
C_IOC = "an IOC remainder cancelled"
C_NONREST = "a NON flow order that traded and rested its residual"
C_FOUR = "four participants"
EAGER_CLASSES = (C_ICE_TRADED, C_TWO_OWNERS)
ALL_CLASSES = (C_ICE_TRADED, C_TWO_OWNERS, C_ICE_BOTH, C_IOC, C_NONREST, C_FOUR)
WRONG_REPORTS = (
    "every trade on the contract (no per-agent filter)",
    "only the trades where the agent bought",
    "only the trades where the agent sold",
)

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C13_anonymity.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION}, {DRAWS} valid draws from seed {SEED0} (at most {MAX_ATTEMPTS} attempts)")
print(f"eager: the first {EAGER_DRAWS} valid draws + the first of each of {len(EAGER_CLASSES)} eager classes")
print(f"coverage classes ({len(ALL_CLASSES)}):")
for _c in ALL_CLASSES:
    print(f"  - {_c}")
print(f"env configuration (section 6 must use exactly this, plus the tapes): {ENV_CONFIG}")
print(f"participant id {PARTICIPANT_ID}; policies {POLICIES}; random AgentOrders: none with p = {P_NOOP}, BUY with p = "
      f"{P_BUY}, restriction {P_EXEC}, quantity <= Q_AGENT_MAX = {Q_AGENT_MAX} qty ticks, GFS, price in mid +- H")
print(f"capacity envelope: {ENVELOPE}; slices per iceberg <= {MAX_SLICES}")
print(f"premise of the participant-view twin (F17): {PREMISE!r}; where it fails, the rule-implied (2r) is asserted, "
      f"and the draw is kept (counted, listed in full in section 7); K2 r4: the remainder shows v1 = min(P - u, R), "
      f"so the rule-implied difference is zero here (F17) and >= 1 such draw (u > 0) is required (F19)")

## 4. Dataset

`make_draw(seed)` below builds world `A`, its twins and the random policy from `numpy.random.Generator(PCG64(seed))` with the generator's public helpers, and returns a generator `Draw` whose `aux_tapes` are the twins (finding F2). In order:

1. **An attempt.** A mid price $m$ (5–300 EUR/MWh), a half-width $H$ (2–20 ticks), 3 or 4 **participants** with distinct random ids in 1–999 plus one **absent id** that places no order. **Resting book:** $N_B \in [4, 10]$ rows, at least one per side, bids at $m - \delta$ and asks at $m + \delta$, $\delta \in [1, H]$ (nothing crosses on entry); each is an iceberg with probability 0.4 (at least one is), else a NON order of 0.1–5 MW. **Order flow:** $N_F \in [3, 10]$ rows, NON or IOC (0.5 each), side random, limit in $[m - H, m + H]$, 0.1–2 MW. Owners are drawn among the participants. With $R = N_B + N_F$ rows the episode has $K = \lceil R/4 \rceil + 1$ env steps.
2. **Iceberg sizes** (finding F4). For an iceberg on side $s$: its peak is $P = U + e$, $e \in [1, 200]$ ticks, with $U$ = the total quantity of all opposite-side flow rows + `Q_AGENT_MAX` $\times K$ (the participant's largest possible order on every step); then a slice count $n \in [2, 8]$ and $T = (n-1)P + r$, $r \in [1, P]$, so $\lceil T/P \rceil = n \le 64$ by construction. No quantity that could ever reach the iceberg can exhaust its visible peak. This bounds what trades **against** a resting iceberg; it does not stop a book-phase iceberg's **entry** from crossing a resting order the participant placed in an earlier step (the participant's random prices span the order-flow band, which straddles the resting book). That is the premise of finding F17, checked in section 7 (K2 r3). Such an entry sweeps at most `Q_AGENT_MAX` $\times K < P$ (the participant's own icebergs' hidden part included), so under the amended item (4) the remainder rests as instance 0 showing $P - f$, as `B`'s NON residual does (K2 r4).
3. **Rows.** `TapeBuilder(OMIEProductConfig())`: one contract (0), distinct random order ids in 1–99 999, tape timestamps $1, 2, \ldots$, every row GFS with `expiry_time = 0`, no price increment, then pad rows.
4. **The acceptance test** (from the reference walk (W) on `A` alone): (W) models every event (no iceberg crosses on entry, no visible peak is exhausted, no FOK); at least one trade; no self-trade (F8); every participant places an order; and **at least one noop view step has an iceberg at the touch** (the best price of its side), so the participant-view twin and the positive control are not vacuous. View step $k \ge 1$ of the noop policy sees the book after $\min(4k, R)$ rows. A rejected attempt is replaced by the next attempt **on the same random stream**, so a seed still names one draw; after 64 rejected attempts the seed is refused.
5. **The twins** (section 1, "The twins"), drawn on the same stream: `B` with a derangement $\sigma$ of the participants and a derangement $\tau$ of all order ids; for each participant $a$, `B@a` with $\sigma_a$ fixing $a$ and deranging the others and $\tau_a$ deranging the order ids of rows not owned by $a$ ($a$'s rows, icebergs included, are copied unchanged); `CTRL` = `B` with one tick added to the quantity of the row that replaces the first iceberg found at the touch. Then the random policy (REL-09: random `AgentOrder`s), on the same PCG64 stream, after the twins. Per step $k = 1, \ldots, K$ the same six draws are always consumed, in this order, whether used or not, so that a seed fixes the stream: (1) $u \sim U[0,1)$, no order (`AgentOrder.none()`) if $u < 0.3$; (2) side BUY if $U[0,1) < 0.5$, else SELL; (3) restriction NON / ICE / IOC / FOK with probabilities 0.40 / 0.30 / 0.15 / 0.15; (4) price tick $p$ uniform in $[m - H, m + H]$ (the order-flow band; the price passed is $p$'s value in EUR/MWh, $0.01 \times$ ticks above 0.00 EUR/MWh, as float32); (5) quantity tick $q$ uniform in $[1, Q_{\mathrm{AGENT\_MAX}}]$ for NON, IOC and FOK, and in $[2, Q_{\mathrm{AGENT\_MAX}}]$ for ICE (its total); (6) peak tick $e$ uniform in $[1, q - 1]$ for ICE, otherwise drawn (from $[1, \max(q - 1, 1)]$) and set to 0. The order is `AgentOrder.create(side, price, q * 0.1, exec_restriction, validity=GFS, expiry_time=0, peak_qty=e * 0.1, ppd=0.0)`; `draw.params["actions"]` records, per step, `[active, side, exec, p, q, e]` in integers ($p$ as the engine's `price_int`). The price band straddles the resting book, so participant orders both aggress (as buyer and as seller) and rest; about 21 % of steps submit a participant iceberg of 2–6 ticks with a smaller peak, so tape flow can regenerate its own hidden part (§4.2, identically in both worlds). Kept out: ICEwPI (`ppd = 0`, F21), GTD (C11's, F14), cancellation (C04's).

The analytic bounds (`tape_capacity_bounds`) of every world are checked against the envelope by `check_envelope`, which refuses the seed (`CapacityBreach`) if any is exceeded. `draw.params` records every drawn value: `mid_int`, `half_width_ticks`, `participants`, `absent_id`, `n_rows`, `n_steps`, `actions`, every order (`oid, phase, side, exec, price_int, qty_int, peak_int, agent`), each iceberg's `peak_bound` and `slices`, the twins' relabellings (`twins`: `keep`, `sigma`, `tau`), `touch_steps` and the control (`ctrl`).

**The reference walk (W)** is defined in this section because the acceptance test uses it; section 5 states it as the known result. It is §6.1 and §6.2 items 2–6 with §4.5: an incoming order meets the opposite side ranked by price, then entry; while it has quantity left and the best opposite entry crosses ($\text{sell} \le \text{buy}$), it trades $\min(\text{left}, \text{resting})$ **at the resting entry's price**; a NON residual rests at its limit, an IOC residual is cancelled. An iceberg rests as an entry of its visible peak (§4.2 "Only the peak size is shown"); since its peak is never exhausted (step 2, checked by (W)), no regeneration rule is needed.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings), checked from the tapes before the run, for every world: the analytic bounds within the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs) and every iceberg within $\lceil T/P \rceil \le 64$ slices; for `A`, `B` and `CTRL` also the env-side bounds that add the participant (F10: per step at most one order, `Q_AGENT_MAX` trades, one resting row and one iceberg slot). After the run, the battery's I8 and the episodes' flags confirm them.

Below: the first draw's worlds `A`, `B` and one `B@a` in readable units beside the integers, the construction digest, the book of `A` and `B` side by side at the first noop view step with an iceberg at the touch (the hatched bar is the hidden remainder `B` does not have), and the envelope actually drawn in the initial seed range.

In [ ]:
import hashlib
import json
from collections import Counter

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_EXPIRY = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_VALIDITY,
                                               synth.COL_EXPIRY)
C_PEAK, C_PPD = synth.COL_PEAK, synth.COL_PPD
ORDER_KEYS = ("oid", "phase", "side", "exec", "price_int", "qty_int", "peak_int", "agent")


class ConstructionRefused(Exception):
    """No attempt of a seed passed the acceptance test (section 4, step 4)."""


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def eur(p_int):
    # Display only: the price in EUR/MWh (§2.3).
    return (int(p_int) - O_TICKS) * PRICE_TICK_EUR


def mw(q_int):
    # Display only: the quantity in MW (§2.3).
    return int(q_int) * QTY_TICK_MW


def ceil_div(a, b):
    return -(-int(a) // int(b))


def n_steps_of(n_rows):
    # Env steps per episode: every real row is replayed, plus one step of pad rows.
    return ceil_div(n_rows, ORDERS_PER_STEP) + 1


def crosses(rest_side, p_rest, p_in):
    # §6.2 item 2: sell price <= buy price, for a resting entry on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def _touch(book):
    # Best bid, best ask, and the icebergs resting at the touch of their side.
    bids = [e["price"] for e in book if e["side"] == BUY]
    asks = [e["price"] for e in book if e["side"] == SELL]
    bb, ba = (max(bids) if bids else None), (min(asks) if asks else None)
    at = sorted(e["oid"] for e in book if e["ice"] and e["price"] == (bb if e["side"] == BUY else ba))
    return [bb, ba, at]


# ---- the reference walk (W): §6.1, §6.2 items 2-6, §4.5, with icebergs as resting peaks (§4.2) -----------------
def walk(tape):
    # Returns the trade rows [buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission], the
    # final book, the icebergs, the IOC remainders, the NON aggressors that rested a residual, the touch after each
    # row, and `trouble`: every event (W) does not model (an iceberg crossing on entry, an exhausted peak, ...).
    book, trades, ioc_cancelled, rested, trouble, after, ice = [], [], {}, [], [], [], {}
    for s, r in enumerate(real_rows(tape)):
        side, p, q = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY])
        oid, ag, ex = int(r[C_OID]), int(r[C_AGENT]), int(r[C_EXEC])
        if ex == ICE:
            peak = int(r[C_PEAK])
            if any(e["side"] != side and crosses(e["side"], e["price"], p) for e in book):
                trouble.append(f"iceberg {oid} would cross on entry")
            if not 1 <= peak < q:
                trouble.append(f"iceberg {oid}: peak {peak} not in [1, total {q})")
            book.append({"side": side, "price": p, "qty": peak, "oid": oid, "agent": ag, "entry": s, "ice": True})
            ice[oid] = {"owner": ag, "side": side, "price": p, "total": q, "peak": peak, "hidden": q - peak}
            after.append(_touch(book))
            continue
        if ex not in (NON, IOC):
            trouble.append(f"oid {oid}: execution restriction {ex} is not drawn")
        left = q
        while left > 0:
            opp = [e for e in book if e["side"] != side and crosses(e["side"], e["price"], p)]
            if not opp:
                break
            best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["entry"]))
            f = min(left, best["qty"])
            if side == BUY:
                trades.append([oid, best["oid"], ag, best["agent"], best["price"], f, s])
            else:
                trades.append([best["oid"], oid, best["agent"], ag, best["price"], f, s])
            left -= f
            best["qty"] -= f
            if best["qty"] == 0:
                book.remove(best)
                if best["ice"]:
                    trouble.append(f"iceberg {best['oid']}'s visible peak exhausted at submission {s}")
        if left > 0 and ex == NON:                                        # §6.2 item 4 / §4.5 NON
            book.append({"side": side, "price": p, "qty": left, "oid": oid, "agent": ag, "entry": s, "ice": False})
            if left < q:
                rested.append(oid)
        elif left > 0 and ex == IOC:                                      # §6.2 item 5 / §4.5 IOC
            ioc_cancelled[oid] = left
        after.append(_touch(book))
    return {"trades": trades, "book": book, "ice": ice, "ioc_cancelled": ioc_cancelled,
            "rested_after_trade": rested, "after": after, "trouble": trouble}


def touch_steps(w, n_rows, n_steps):
    # Observation k (k = 1..n_steps, after step k of the noop policy) sees the book after min(4k, R) rows.
    out = []
    for k in range(1, n_steps + 1):
        at = w["after"][min(ORDERS_PER_STEP * k, n_rows) - 1][2]
        if at:
            out.append([k, at])
    return out


# ---- the twins (section 4, "The twins") ---------------------------------------------------------------------------
def derangement(rng, items):
    # A random bijection of `items` onto itself with no fixed point (len(items) >= 2).
    items = [int(x) for x in items]
    if len(items) < 2:
        raise ValueError("a derangement needs at least two items")
    while True:
        perm = [items[int(i)] for i in rng.permutation(len(items))]
        if all(a != b for a, b in zip(items, perm)):
            return dict(zip(items, perm))


def twin_orders(orders, keep, sigma, tau):
    # Rows owned by `keep` are copied unchanged; every other row gets its agent and order id relabelled, and an
    # iceberg becomes a NON order of its peak (hidden remainder removed, finding F5). Same position, price, side.
    out = []
    for o in orders:
        o2 = dict(o)
        if keep is None or int(o["agent"]) != int(keep):
            o2["agent"] = int(sigma[str(o["agent"])])
            o2["oid"] = int(tau[str(o["oid"])])
            if int(o["exec"]) == ICE:
                o2.update(exec=NON, qty_int=int(o["peak_int"]), peak_int=0)
        out.append(o2)
    return out


def relabel(trades, sigma, tau):
    # (W)'s rows under a relabelling (identity for keys absent from the maps).
    s = lambda a: int(sigma.get(str(a), a))   # noqa: E731
    t = lambda o: int(tau.get(str(o), o))     # noqa: E731
    return [[t(b), t(so), s(ba), s(sa), p, q, sub] for b, so, ba, sa, p, q, sub in trades]


def _tape(orders):
    tb = synth.TapeBuilder(CONFIG)
    for o in orders:
        tb.add(side=o["side"], price_int=o["price_int"], qty_int=o["qty_int"], agent_id=o["agent"],
               exec_restriction=o["exec"], validity=GFS, expiry_time=0, peak_int=o["peak_int"], ppd_int=0,
               order_id=o["oid"])
    return tb.build()


def _rint(rng, lo_hi):
    lo, hi = lo_hi
    return int(rng.integers(lo, hi + 1))


def _attempt(rng):
    # One attempt of step 2: the orders of world A (with oids) and the attempt's drawn values.
    mid = O_TICKS + _rint(rng, MID)
    H = _rint(rng, HALF_WIDTH)
    A = _rint(rng, PARTICIPANTS)
    pick = rng.choice(np.arange(ID_POOL[0], ID_POOL[1] + 1), size=A + 1, replace=False)
    participants, absent = [int(x) for x in pick[:A]], int(pick[A])
    nb, nf = _rint(rng, N_BOOK), _rint(rng, N_FLOW)
    n_rows = nb + nf
    steps = n_steps_of(n_rows)
    sides = [BUY, SELL] + [BUY if float(rng.random()) < 0.5 else SELL for _ in range(nb - 2)]
    book = []
    for i in rng.permutation(nb):
        side = sides[int(i)]
        d = _rint(rng, (1, H))
        book.append({"phase": "book", "side": side, "price_int": mid - d if side == BUY else mid + d,
                     "ice": float(rng.random()) < P_ICE, "agent": participants[int(rng.integers(A))]})
    if not any(o["ice"] for o in book):
        book[int(rng.integers(nb))]["ice"] = True
    flow = []
    for _ in range(nf):
        side = BUY if float(rng.random()) < 0.5 else SELL
        flow.append({"phase": "flow", "side": side, "price_int": mid + _rint(rng, (-H, H)),
                     "exec": IOC if float(rng.random()) < P_IOC else NON, "qty_int": _rint(rng, Q_FLOW),
                     "peak_int": 0, "agent": participants[int(rng.integers(A))]})
    for o in book:
        if o.pop("ice"):
            # Finding F4: the visible peak exceeds every quantity that could ever trade against it.
            bound = sum(f["qty_int"] for f in flow if f["side"] != o["side"]) + Q_AGENT_MAX * steps
            peak = bound + _rint(rng, PEAK_EXTRA)
            n = _rint(rng, SLICES)
            o.update(exec=ICE, peak_int=peak, qty_int=(n - 1) * peak + _rint(rng, (1, peak)), peak_bound=bound,
                     slices=n)
        else:
            o.update(exec=NON, qty_int=_rint(rng, Q_BOOK), peak_int=0)
    orders = book + flow
    oids = rng.choice(np.arange(OID_POOL[0], OID_POOL[1] + 1), size=n_rows, replace=False)
    for o, oid in zip(orders, oids):
        o["oid"] = int(oid)
    return {"mid_int": int(mid), "half_width_ticks": H, "participants": participants, "absent_id": absent,
            "n_steps": steps, "orders": orders}


def _reject_reason(att, w):
    # Step 4, the acceptance test on world A's walk; None = accepted.
    if w["trouble"]:
        return "walk trouble: " + w["trouble"][0]
    if not w["trades"]:
        return "no trade"
    if any(t[2] == t[3] for t in w["trades"]):
        return "a self-trade (not drawn, finding F8)"
    owners = {int(o["agent"]) for o in att["orders"]}
    if owners != set(att["participants"]):
        return "a participant places no order"
    if not touch_steps(w, len(att["orders"]), att["n_steps"]):
        return "no iceberg at the touch at any noop view step"
    return None


def draw_agent_orders(rng, mid, H, n_steps):
    # Step 5 (REL-09): one [active, side, exec, p, q, e] per step; the same six draws are always consumed, in order.
    out = []
    for _ in range(int(n_steps)):
        u, s, x = float(rng.random()), float(rng.random()), float(rng.random())
        ex = next(e for cut, e in EXEC_CUTS if x < cut)
        p = int(rng.integers(int(mid) - int(H), int(mid) + int(H) + 1))
        q = int(rng.integers(2 if ex == ICE else 1, Q_AGENT_MAX + 1))
        e = int(rng.integers(1, max(q - 1, 1) + 1))
        out.append([int(u >= P_NOOP), BUY if s < P_BUY else SELL, ex, p, q, e if ex == ICE else 0])
    return out


def agent_order(a):
    # One recorded [active, side, exec, p, q, e] as the AgentOrder the participant submits (GFS, no ICEwPI).
    active, side, ex, p, q, e = (int(v) for v in a)
    if not active:
        return AgentOrder.none()
    return AgentOrder.create(side, np.float32((p - O_TICKS) * PRICE_TICK_EUR), np.float32(q * QTY_TICK_MW), ex,
                             validity=GFS, expiry_time=0, peak_qty=np.float32(e * QTY_TICK_MW), ppd=0.0)


def make_draw(seed):
    # One C13 draw (steps 1-5). Raises ConstructionRefused, or synth.CapacityBreach on an analytic-bound breach.
    rng = np.random.Generator(np.random.PCG64(int(seed)))
    rejected = Counter()
    for attempt in range(MAX_TRIES):
        att = _attempt(rng)
        orders = att["orders"]
        tape_a = _tape(orders)
        w = walk(tape_a)
        why = _reject_reason(att, w)
        if why is None:
            break
        rejected[why.split(":")[0]] += 1
    else:
        raise ConstructionRefused(f"seed {seed}: none of {MAX_TRIES} attempts accepted: {dict(rejected)}")
    parts = att["participants"]
    # The participant-view twin B: every tape participant is foreign to the Session participant.
    twins = {"B": {"keep": None,
                   "sigma": {str(a): b for a, b in derangement(rng, parts).items()},
                   "tau": {str(a): b for a, b in derangement(rng, [o["oid"] for o in orders]).items()}}}
    # The observer twins B@a: a and its rows kept; every other participant and order id relabelled.
    for a in parts:
        others = [p for p in parts if p != a]
        foreign = [o["oid"] for o in orders if int(o["agent"]) != a]
        sigma = {str(a): a, **{str(x): y for x, y in derangement(rng, others).items()}}
        tau = {str(x): y for x, y in derangement(rng, foreign).items()}
        twins[f"B@{a}"] = {"keep": a, "sigma": sigma, "tau": tau}
    aux = {name: _tape(twin_orders(orders, tw["keep"], tw["sigma"], tw["tau"])) for name, tw in twins.items()}
    # The positive control: world B plus one tick on the visible peak of the first iceberg at the touch.
    ts = touch_steps(w, len(orders), att["n_steps"])
    ctrl_oid_a = int(ts[0][1][0])
    ctrl_row = next(i for i, o in enumerate(orders) if int(o["oid"]) == ctrl_oid_a)
    ctrl_orders = twin_orders(orders, None, twins["B"]["sigma"], twins["B"]["tau"])
    ctrl_orders[ctrl_row]["qty_int"] += 1
    aux["CTRL"] = _tape(ctrl_orders)
    # The random policy's AgentOrders (drawn after acceptance and the twins, on the same stream; step 5).
    actions = draw_agent_orders(rng, att["mid_int"], att["half_width_ticks"], att["n_steps"])
    params = {"seed": int(seed), "attempt": int(attempt), "rejected_attempts": dict(rejected),
              "mid_int": att["mid_int"], "mid_eur": round(eur(att["mid_int"]), 2),
              "half_width_ticks": att["half_width_ticks"], "participants": parts, "absent_id": att["absent_id"],
              "n_rows": len(orders), "n_steps": att["n_steps"], "actions": actions,
              "orders": [{k: int(o[k]) if k != "phase" else o[k] for k in ORDER_KEYS} for o in orders],
              "icebergs": {str(o["oid"]): {"peak_bound": int(o["peak_bound"]), "slices": int(o["slices"])}
                           for o in orders if o["exec"] == ICE},
              "twins": twins, "touch_steps": ts,
              "ctrl": {"row": ctrl_row, "oid_A": ctrl_oid_a, "oid_B": int(twins["B"]["tau"][str(ctrl_oid_a)]),
                       "step": int(ts[0][0])}}
    cb = synth.tape_capacity_bounds(tape_a)
    synth.check_envelope(ENVELOPE, **cb)                                       # CapacityBreach -> refused
    for t in aux.values():
        synth.check_envelope(ENVELOPE, **synth.tape_capacity_bounds(t))
    return synth.Draw(construction=CONSTRUCTION, seed=int(seed), tape=tape_a, params=params, expected={},
                      aux_tapes=types.MappingProxyType(aux), hop_bound=int(cb["hops"]),
                      capacity_bounds=types.MappingProxyType({k: int(cb[k]) for k in synth.AXES}),
                      oracle_kind="exact-parametric", meta={"built_in": "C13 notebook section 4"})


def world_tape(draw, world):
    # "A" is the draw's tape; "B", "B@<a>" and "CTRL" are its aux tapes.
    return draw.tape if world == "A" else draw.aux_tapes[world]


def observer_worlds(draw):
    return {int(a): f"B@{a}" for a in draw.params["participants"]}


DRAWN = {}       # seed -> Draw, or None when the builder refused the seed
REFUSED = {}     # seed -> the refusal message


def prepare(seed):
    # Build one seed (the run loop of section 7 calls this for a re-draw too).
    try:
        d = make_draw(seed)
    except (synth.CapacityBreach, ConstructionRefused) as exc:
        d = None
        REFUSED[seed] = f"{type(exc).__name__}: {exc}"
    DRAWN[seed] = d
    return d


for _s in range(SEED0, SEED0 + DRAWS):
    prepare(_s)


def _digest_or_refused(s):
    try:
        return make_draw(s).digest()
    except (synth.CapacityBreach, ConstructionRefused):
        return "refused"


C13_DIGEST = hashlib.blake2b("".join(_digest_or_refused(s) for s in range(8)).encode("ascii"),
                             digest_size=16).hexdigest()


def tape_table(tape, title, phases=None):
    rows = ["| s | oid | phase | side | exec | price (EUR/MWh) | qty (MW) | peak (MW) | hidden (MW) | price_int | "
            "qty_int | peak_int | agent | tape ts |",
            "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(tape)):
        ice = int(r[C_EXEC]) == ICE
        rows.append(f"| {k} | {int(r[C_OID])} | {(phases or {}).get(k, '')} | {side_name(r[C_SIDE])} | "
                    f"{EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | {eur(r[C_PRICE]):.2f} | {mw(r[C_QTY]):.1f} | "
                    f"{(f'{mw(r[C_PEAK]):.1f}' if ice else '')} | "
                    f"{(f'{mw(int(r[C_QTY]) - int(r[C_PEAK])):.1f}' if ice else '')} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_PEAK])} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST = next((s for s in range(SEED0, SEED0 + DRAWS) if DRAWN[s] is not None), None)
if FIRST is None:
    print("every seed of the initial range was refused by the builder")
else:
    _d = DRAWN[FIRST]
    _p = _d.params
    _ph = {k: o["phase"] for k, o in enumerate(_p["orders"])}
    display(Markdown(tape_table(_d.tape, f"World A — seed {FIRST} (digest `{_d.digest()}`)", _ph)))
    display(Markdown(tape_table(world_tape(_d, "B"), f"World B — its participant-view twin "
                                                     f"(agents {_p['twins']['B']['sigma']})", _ph)))
    _a0 = _p["participants"][0]
    display(Markdown(tape_table(world_tape(_d, f"B@{_a0}"), f"World B@{_a0} — the trade-report twin of observer "
                                                             f"{_a0} (its own rows kept)", _ph)))
    print(f"seed {FIRST}: mid {_p['mid_eur']:.2f} EUR/MWh (int {_p['mid_int']}), half-width {_p['half_width_ticks']} "
          f"ticks, participants {_p['participants']}, absent id {_p['absent_id']}, {_p['n_rows']} rows, "
          f"{_p['n_steps']} env steps, accepted at attempt {_p['attempt']}")
    print(f"  icebergs (peak bound, slices): {_p['icebergs']}")
    print(f"  icebergs at the touch, per noop view step: {_p['touch_steps']}")
    print(f"  positive control: {_p['ctrl']}; random AgentOrders [active, side, exec, p, q, e]: {_p['actions']}")
    print(f"  capacity bounds of A: {dict(_d.capacity_bounds)}")
print(f"construction digest over seeds 0-7 (construction_digest formula): {C13_DIGEST}")
print(f"seeds refused by the builder in the initial range: {len(REFUSED)}" + (f" ({sorted(REFUSED)})" if REFUSED else ""))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
if FIRST is not None:
    _d = DRAWN[FIRST]
    _k = _d.params["ctrl"]["step"]
    _n = min(ORDERS_PER_STEP * _k, _d.params["n_rows"])
    for ax, world in zip(axes, ("A", "B")):
        # The book of the noop policy at the first view step with an iceberg at the touch, from (W).
        _w = walk(real_rows(world_tape(_d, world))[:_n])
        _ice = _w["ice"]
        for e in _w["book"]:
            sgn = 1 if e["side"] == BUY else -1
            col = "tab:green" if sgn > 0 else "tab:red"
            ax.barh(eur(e["price"]), sgn * mw(e["qty"]), height=0.006 * _d.params["half_width_ticks"], color=col,
                    edgecolor="black", lw=0.3)
            if e["oid"] in _ice:
                h = _ice[e["oid"]]["hidden"]
                ax.barh(eur(e["price"]), sgn * mw(h), left=sgn * mw(e["qty"]),
                        height=0.006 * _d.params["half_width_ticks"], color="none", edgecolor=col, hatch="///", lw=0.6)
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("MW: buy right, sell left (solid: visible; hatched: hidden remainder)", fontsize=7)
        ax.set_title(f"seed {FIRST}, world {world}: the book at noop view step {_k}", fontsize=8)
    axes[0].set_ylabel("limit price (EUR/MWh)", fontsize=7)
plt.tight_layout()
plt.show()

_ok = [d for d in DRAWN.values() if d is not None]
if _ok:
    _env = {}
    _ords = [o for d in _ok for o in d.params["orders"]]
    _ices = [o for o in _ords if o["exec"] == ICE]
    _env["rows per draw"] = [d.params["n_rows"] for d in _ok]
    _env["env steps per draw"] = [d.params["n_steps"] for d in _ok]
    _env["participants per draw"] = [len(d.params["participants"]) for d in _ok]
    _env["icebergs per draw"] = [len(d.params["icebergs"]) for d in _ok]
    _env["iceberg peak (MW)"] = [mw(o["peak_int"]) for o in _ices]
    _env["iceberg total (MW)"] = [mw(o["qty_int"]) for o in _ices]
    _env["slices ceil(total/peak)"] = [ceil_div(o["qty_int"], o["peak_int"]) for o in _ices]
    _env["noop view steps with an iceberg at the touch"] = [len(d.params["touch_steps"]) for d in _ok]
    _env["mid price (EUR/MWh)"] = [d.params["mid_eur"] for d in _ok]
    _env["attempts used per seed"] = [d.params["attempt"] + 1 for d in _ok]
    display(Markdown("**The envelope actually drawn in the initial seed range** (empirical test ranges, never bounds: "
                     "finding F14)\n\n| quantity | min | max |\n|---|---|---|\n"
                     + "\n".join(f"| {k} | {min(v):g} | {max(v):g} |" for k, v in _env.items())))

## 5. Known result

**Notation.** Integer ticks throughout. A trade row is `[buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission]`. An iceberg $j$ has total $T_j$, peak $P_j$ and hidden remainder $h_j = T_j - P_j \ge 1$ while no slice is exhausted. $\sigma$ relabels agents and $\tau$ relabels order ids (section 4); `B` uses $(\sigma, \tau)$, `B@a` uses $(\sigma_a, \tau_a)$ with $\sigma_a(a) = a$ and $\tau_a$ fixing $a$'s orders.

**(1) The engine, both worlds** (§6.1, §6.2 items 2–6, §4.5, §4.2; D3-08). (W) pins every trade row of `A`, in submission order and within a submission in walk order, with an iceberg trading as a resting order of its visible peak. Since no peak is exhausted (F4), `B`'s visible book equals `A`'s at every instant, and matching is ownership-blind (§6.6) and keyed by price and timestamp (§6.1, §5.2), so

$$\mathrm{trades}(B) = \mathrm{relabel}_{\sigma,\tau}\big(\mathrm{trades}(A)\big),$$

checked on the tapes in the precondition and asserted on the engine. Fates: each order's filled quantity; each resting order's visible quantity; $h_j$ for each iceberg of `A` (none in `B`); each IOC residual cancelled as `ioc_remainder`; nothing evicted.

**(2) The participant-view twin** (§7.1, §7.3, §9.2 "LOB per contract", "Best bid / best ask"), **where the premise holds**: no tape iceberg fills a resting participant order on entry. That is, no trade of the episode, in `A` or in `B`, has a tape iceberg (in `B`, its replacement row) as aggressor (finding F17). For each policy $\pi \in \{\texttt{noop}, \texttt{random}\}$ and every step $k = 0, \ldots, K$ ($K$ = `n_steps`, step 0 = reset):

$$\mathrm{tob}_k(A, \pi) \equiv \mathrm{tob}_k(B, \pi), \qquad \mathrm{lob}_k(A, \pi) \equiv \mathrm{lob}_k(B, \pi),$$

and for every step $k = 1, \ldots, K$ and at the end:

$$\mathrm{fill}_k(A,\pi) \equiv \mathrm{fill}_k(B,\pi), \quad d_k(A,\pi) = d_k(B,\pi), \quad \mathrm{own}(A,\pi) \equiv \mathrm{own}(B,\pi),$$

Here $\mathrm{tob}$ is `top_of_book`, $\mathrm{lob}$ is `get_anonymous_lob`, $\mathrm{fill}$ is the `FillReport`, and $\mathrm{own}$ is the participant's own trades, own orders, `agent_cash` and `agent_inventory` at the end. $\equiv$ is bit-identity. The two worlds differ only in what the participant may not see: the foreign hidden remainders and the foreign identifiers.

**(2r) Where the premise fails: the rule-implied display** (§4.2.1 Rule 2, *Incoming iceberg* (1), (2), (4), (6), amended 2026-10-20; OQ-34 resolution, display C3; OMIE Rules Annex §4.4, `:10256-10261`; K2 r3, re-derived at K2 r4, finding F17). A tape iceberg $j$ at tape row $s_j$, price $b_j$, replayed at env step $k_j = \lfloor s_j / 4 \rfloor + 1$, crosses a resting order of the participant when it enters and aggresses with its **total** quantity, as a chain of instances of size $P_j$, all at limit $b_j$ ($d = 0$). Its swept quantity $M_j$ is $f_j$, the participant's total fill on that entry (the trades whose aggressor is $j$, all with the participant as passive side), and $1 \le f_j \le f_{\max} = $ `Q_AGENT_MAX` $\times K < P_j$ (the peak bound of section 4). The participant's resting quantity that $j$ sweeps may include the hidden part of a participant iceberg (Rule 2 multi-slice: its slices are swept in turn). $f_j$ counts all of it, and $f_j \le f_{\max} < P_j$ still holds. So, with $R_j = T_j - f_j$,

$$k_e = \lfloor M_j / P_j \rfloor = 0, \qquad u_j = M_j - k_e P_j = f_j \ge 1, \qquad v_1 = \min(P_j - u_j, R_j) = P_j - f_j,$$

since $R_j \ge P_j + 1 - f_j$. `A` rests $j$ as instance 0 at $b_j$, showing $v_1$ and hiding $R_j - v_1 = h_j$; `B` rests the NON residual $P_j - f_j$ of $\tau(j)$, at its entry timestamp, as `A`'s remainder keeps its own (§5.2, §6.2 item 4). Later fills are the same in both worlds and exhaust neither, so at the end of the episode

$$\mathrm{vis}_A(j) - \mathrm{vis}_B(\tau(j)) = v_1 - (P_j - f_j) = 0, \qquad \mathrm{hid}_A(j) = R_j - v_1 = h_j .$$

Asserted, exactly, in integer ticks: every entry trade of $j$ in `A` has the participant as its passive side and $1 \le f_j \le f_{\max} < P_j$; $\tau(j)$'s entry trades in `B` total the same $f_j$, all with the participant as passive side; the two identities above on the episode's final book; `A`'s final book holds exactly one resting row of $j$, at $b_j + k_e\,d = b_j$; and, since nothing visible differs, the participant view of every step and at the end (as in (2)) **bit-identical** between `A` and `B`. `surfacing(entry, M)` below computes the rule-implied values for any $M$ (general item (4)); on this construction they reduce to the line above. The draw is **not** discarded. (r3 used the retired display $\min(P, R)$: `A` then showed $\min(f_j, h_j)$ more than `B` and the observation was asserted only before the first entry step; with $u_j > 0$ on every such draw, an engine still displaying $\min(P, R)$ fails this assertion by $f_j$ ticks, finding F19.)

**(3) The positive control.** $\mathrm{tob}(\mathtt{CTRL}, \texttt{noop}) \ne \mathrm{tob}(B, \texttt{noop})$ on at least one step: one extra visible tick on an iceberg that is at the touch (from step `ctrl.step` on, by (W)) must be visible in `q_bid` or `q_ask`, and in the matching level of `get_anonymous_lob`. Without it, (2) could pass on a reader that sees nothing.

**(4) The trade-report partition** (§7.2, §9.2 "Trade book"). For every id $a$ among the participants, the absent id and the participant's id (`AGENT_TRADER_ID`), with the leg $\ell(t, a)$ = `[price_int, qty_int]` of trade $t$ when $a$ is its buyer or its seller (no side: correction J4, finding F16):

$$\mathrm{legs}\big(\mathrm{report}_A(a)\big) = \mathrm{sort}\,[\ell(t, a) : t \in \mathrm{trades}(A),\ a \in \{\mathrm{buyer}(t), \mathrm{seller}(t)\}],$$

empty for the absent id and the participant (the engine worlds carry no `Session` participant). The union over all ids is every trade's leg exactly twice, once from its buyer's report and once from its seller's, and the reports hold $2\,\lvert\mathrm{trades}(A)\rvert$ rows in all (no duplication, no self-trade drawn).

**(5) The trade-report twin** (§7.2 *"Counterparty identity is not disclosed"*, F7): for every participant $a$, $\mathrm{report}_A(a) \equiv \mathrm{report}_{B@a}(a)$.

**(6) The ownership side-condition** (§9.2 "Own orders", §7.3, F6). For every iceberg $j$ of `A` with owner $o_j$: the engine's final book holds exactly one resting row of $j$, at its price, owned by $o_j$, of visible quantity $v_j = P_j - f_j$ ($f_j$ its filled quantity), and hidden $h_j$; so the owner's full remaining quantity $v_j + h_j = T_j - f_j$ is held. If section 6 exposes `own_orders_view`, it shows each participant exactly its own resting orders, each iceberg at its full remaining quantity $T_j - f_j$: assertion (6′), expected to pass now that `OMIE-OWN-ORDERS-ICE-QTY-1` has merged (S6).

**(6″) Ownership side-condition, participant side** (§9.2 "Own orders", finding F6). In every `random` episode, `get_agent_orders(final_book, AGENT_TRADER_ID)` lists exactly the participant's resting orders on the episode's final book (each resting row whose owner is `AGENT_TRADER_ID`, once), each at its full remaining quantity, visible + hidden (for a participant iceberg, its visible slice plus its hidden remainder), and the list is bit-identical between `A` and `B`. At every price where a participant iceberg rests with a hidden part, `get_anonymous_lob` shows the sum of the **visible** quantities resting there, never the hidden part (finding F23).

**(7) The twin stayed a twin.** At the end of every `A` episode, each tape iceberg's hidden quantity is still $h_j$, an iceberg of (2r) included (its rule-implied $R_j - v_1$ is $h_j$, K2 r4); `B` and `CTRL` hold no hidden quantity of a tape order. The participant's own icebergs are excluded: they may legitimately regenerate, identically in both worlds (finding F4).

**Discrimination** (rule 1): from the tape alone, three wrong reports — every trade on the contract once, only the trades the participant bought in, only those it sold in — each differ from (4), as sorted `[price_int, qty_int]` legs, for some participant on every valid draw (each trade leaves out at least one of the ≥ 3 participants, and each trade has a buyer and a seller among them).

The code below derives everything from the tapes through (W). It never reads `draw.expected`, and it never runs the engine. (2r) is parametric in $f_j$, which section 7 reads from the run: `surfacing(entry, M)` gives the rule-implied values for one iceberg. Below the code, every iceberg of the initial seed range is checked analytically for every possible $f \in [1, f_{\max}]$: $k_e = 0$, $u = f$, a zero visible gap and $\mathrm{hid}_A = h_j$ (the retired display would give a gap of $f$ on each).

In [ ]:
def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def engine_expectation(tape):
    # (1): the trades and per-order fates of one world, from (W).
    rows = real_rows(tape)
    oids = [int(r[C_OID]) for r in rows]
    w = walk(tape)
    filled = {o: 0 for o in oids}
    for t in w["trades"]:
        filled[t[0]] += t[5]
        filled[t[1]] += t[5]
    return {
        "accepted": sorted(oids), "rejected": [],
        "trades": w["trades"],
        "filled": {str(o): filled[o] for o in sorted(oids) if filled[o]},
        "still_resting": {str(e["oid"]): e["qty"] for e in sorted(w["book"], key=lambda e: e["oid"])},
        "hidden_remaining": {str(o): v["hidden"] for o, v in sorted(w["ice"].items())},
        "cancelled": {str(o): [["ioc_remainder", q]] for o, q in sorted(w["ioc_cancelled"].items())},
        "evicted": {},
    }


def surfacing(entry, M):
    # (2r), §4.2.1 Rule 2 Incoming iceberg (1), (4), amended 2026-10-20 (display C3, OQ-34): iceberg `entry` (an
    # `ice_entry` value, d = 0) entered with total T and swept M ticks (in C13: M = f, the participant's resting quantity);
    # its remainder R = T - M rests as instance k_e = floor(M / P) at b + k_e d = b, showing v1 = min(P - u, R) with
    # u = M - k_e P, and hiding R - v1. B's NON row of the peak rests P - M (meaningful for M < P, the only case the
    # construction admits). `retired_C1` is r3's display min(P, R), recorded for comparison and never asserted.
    P, T, h, M = int(entry["peak"]), int(entry["total"]), int(entry["hidden"]), int(M)
    R = T - M
    k_e = M // P
    u = M - k_e * P
    v1 = min(P - u, R)
    return {"M": M, "R": R, "k_e": k_e, "u": u, "shown_A": v1, "rests_B": P - M, "visible_gap": v1 - (P - M),
            "hidden_A": R - v1, "rest_price_A": int(entry["price"]), "hidden_unchanged": (R - v1) == h,
            "retired_C1": {"shown_A": min(P, R), "visible_gap": min(P, R) - (P - M)}}


def legs(trades, agent):
    # (4): the legs [price_int, qty_int] of `agent` (no side: correction J4, finding F16), sorted. No self-trade is drawn.
    return sorted([int(p), int(q)] for _b, _s, b_ag, s_ag, p, q, _sub in trades if int(agent) in (int(b_ag), int(s_ag)))


def expectation(draw):
    # Everything section 7 compares for one draw.
    p = draw.params
    wA = walk(draw.tape)
    t = wA["trades"]
    parts = [int(a) for a in p["participants"]]
    report_ids = parts + [int(p["absent_id"]), PARTICIPANT_ID]
    reports = {str(a): legs(t, a) for a in report_ids}
    union = sorted([[int(x[4]), int(x[5])] for x in t] * 2)          # every trade once per counterparty
    vis = {e["oid"]: e["qty"] for e in wA["book"]}
    ownership = {str(j): {"owner": v["owner"], "price_int": v["price"], "visible": vis.get(j, 0),
                          "hidden": v["hidden"], "full_remaining": vis.get(j, 0) + v["hidden"]}
                 for j, v in sorted(wA["ice"].items())}
    own_orders = {str(a): {str(e["oid"]): e["qty"] + (wA["ice"][e["oid"]]["hidden"] if e["ice"] else 0)
                           for e in sorted(wA["book"], key=lambda e: e["oid"]) if e["agent"] == a} for a in parts}
    # discrimination of the three wrong reports (rule 1)
    every = sorted([int(x[4]), int(x[5])] for x in t)
    bought = {str(a): sorted([int(x[4]), int(x[5])] for x in t if int(x[2]) == a) for a in parts}
    sold = {str(a): sorted([int(x[4]), int(x[5])] for x in t if int(x[3]) == a) for a in parts}
    disc = {
        WRONG_REPORTS[0]: any(every != reports[str(a)] for a in parts),
        WRONG_REPORTS[1]: any(bought[str(a)] != reports[str(a)] for a in parts),
        WRONG_REPORTS[2]: any(sold[str(a)] != reports[str(a)] for a in parts),
    }
    ice_sides = {v["side"] for v in wA["ice"].values()}
    traded_ice = any((x[0] in wA["ice"]) or (x[1] in wA["ice"]) for x in t)
    classes = []
    if traded_ice:
        classes.append(C_ICE_TRADED)
    if len({v["owner"] for v in wA["ice"].values()}) >= 2:
        classes.append(C_TWO_OWNERS)
    if ice_sides == {BUY, SELL}:
        classes.append(C_ICE_BOTH)
    if wA["ioc_cancelled"]:
        classes.append(C_IOC)
    if wA["rested_after_trade"]:
        classes.append(C_NONREST)
    if len(parts) == 4:
        classes.append(C_FOUR)
    tw = p["twins"]
    exB = engine_expectation(world_tape(draw, "B"))
    # (2r): each iceberg's entry row and step, its numbers and its replacement row in B (finding F17)
    row_of = {int(r[C_OID]): k for k, r in enumerate(real_rows(draw.tape))}
    ice_entry = {str(j): {"side": int(v["side"]), "price": int(v["price"]), "owner": int(v["owner"]),
                          "peak": int(v["peak"]), "total": int(v["total"]),
                          "hidden": int(v["hidden"]), "row": row_of[int(j)],
                          "entry_step": row_of[int(j)] // ORDERS_PER_STEP + 1, "oid_B": int(tw["B"]["tau"][str(j)])}
                 for j, v in sorted(wA["ice"].items())}
    return {
        "A": engine_expectation(draw.tape), "B": exB,
        "B_relabelled_from_A": relabel(t, tw["B"]["sigma"], tw["B"]["tau"]),
        "reports": reports, "union": union, "n_report_rows": 2 * len(t),
        "ownership": ownership, "own_orders": own_orders,
        "report_ids": report_ids, "observers": parts, "ice_oids_A": sorted(wA["ice"]), "ice_entry": ice_entry,
        "hidden_A": {str(j): v["hidden"] for j, v in sorted(wA["ice"].items())},
        "oids": {w: sorted(int(r[C_OID]) for r in real_rows(world_tape(draw, w))) for w in ("A", "B", "CTRL")},
        "n_steps": int(p["n_steps"]), "n_obs": int(p["n_steps"]) + 1,
        "disc": disc, "classes": classes, "draw_classes": [c for c in classes if c in EAGER_CLASSES],
        "expects_trade": True,
    }


# ---- the dataset precondition (section 4) ------------------------------------------------------------------------
def precondition(draw):
    # Returns the list of departures (empty: the draw is as section 4 describes).
    bad, t, p = [], np.asarray(draw.tape), draw.params
    rows = real_rows(t)
    try:
        ords = p["orders"]
        parts = [int(a) for a in p["participants"]]
        if len({int(x[C_CONTRACT]) for x in rows}) != 1:
            bad.append("more than one contract")
        ts = [int(x[C_TS]) for x in rows]
        if any(b <= a for a, b in zip(ts, ts[1:])):
            bad.append(f"tape timestamps {ts} not strictly increasing")
        # Early-done ruling (2026-10-26): every env world of a draw carries the same timestamp column, so the
        # span-scaled clock (and a done that freezes the session) reaches each world at the same step.
        ts_cols = {w: np.asarray(world_tape(draw, w))[:, C_TS].tolist() for w in ("A", "B", "CTRL")
                   if w == "A" or w in draw.aux_tapes}
        if len({tuple(v) for v in ts_cols.values()}) != 1:
            bad.append("worlds A, B and CTRL do not share one timestamp column (early-done ruling)")
        oids = [int(x[C_OID]) for x in rows]
        if len(set(oids)) != len(oids) or not all(OID_POOL[0] <= o <= OID_POOL[1] for o in oids):
            bad.append("order ids not distinct or outside the pool")
        if not PARTICIPANTS[0] <= len(parts) <= PARTICIPANTS[1] or len(set(parts)) != len(parts) \
                or PARTICIPANT_ID in parts or int(p["absent_id"]) in parts \
                or not all(ID_POOL[0] <= a <= ID_POOL[1] for a in parts + [int(p["absent_id"])]):
            bad.append(f"participants {parts} / absent id {p['absent_id']} not as drawn")
        p_rows = [[int(o["oid"]), int(o["side"]), int(o["exec"]), int(o["price_int"]), int(o["qty_int"]),
                   int(o["peak_int"]), int(o["agent"])] for o in ords]
        t_rows = [[int(x[C_OID]), int(x[C_SIDE]), int(x[C_EXEC]), int(x[C_PRICE]), int(x[C_QTY]), int(x[C_PEAK]),
                   int(x[C_AGENT])] for x in rows]
        if p_rows != t_rows:
            bad.append("params != tape")
        for x in rows:
            o = int(x[C_OID])
            if int(x[C_VALIDITY]) != GFS or int(x[C_EXPIRY]) != 0 or int(x[C_PPD]) != 0:
                bad.append(f"oid {o}: not a plain GFS row (validity, expiry, increment)")
            if not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not 1 <= int(x[C_QTY]) <= QTY_INT_MAX:
                bad.append(f"oid {o}: price or quantity outside §2.3")
            if int(x[C_EXEC]) == FOK:
                bad.append(f"oid {o}: a FOK (none is drawn, finding F4)")
        phases = [o["phase"] for o in ords]
        nb = phases.count("book")
        if phases != ["book"] * nb + ["flow"] * (len(ords) - nb):
            bad.append(f"phases out of order: {phases}")
        mid, H = int(p["mid_int"]), int(p["half_width_ticks"])
        book, flow = ords[:nb], ords[nb:]
        if any(not (0 < mid - int(o["price_int"]) <= H if int(o["side"]) == BUY else 0 < int(o["price_int"]) - mid <= H)
               for o in book):
            bad.append("a resting-book row not strictly on its side of the mid within the half-width")
        if any(int(o["exec"]) not in (NON, IOC) or not -H <= int(o["price_int"]) - mid <= H
               or not Q_FLOW[0] <= int(o["qty_int"]) <= Q_FLOW[1] for o in flow):
            bad.append("a flow row not NON/IOC, or outside mid +- H, or outside the flow quantity range")
        n_steps = int(p["n_steps"])
        if n_steps != n_steps_of(len(ords)):
            bad.append(f"n_steps {n_steps} != ceil(R/4) + 1")
        ices = [o for o in book if int(o["exec"]) == ICE]
        if not ices:
            bad.append("no iceberg")
        for o in ices:
            opp = sum(int(f["qty_int"]) for f in flow if int(f["side"]) != int(o["side"]))
            bound = opp + Q_AGENT_MAX * n_steps
            sl = ceil_div(o["qty_int"], o["peak_int"])
            if not int(o["peak_int"]) > bound:
                bad.append(f"iceberg {o['oid']}: peak {o['peak_int']} <= bound {bound} (finding F4)")
            if not SLICES[0] <= sl <= SLICES[1] or sl > MAX_SLICES:
                bad.append(f"iceberg {o['oid']}: {sl} slices outside [{SLICES[0]}, {SLICES[1]}]")
        acts = [[int(v) for v in a] for a in p["actions"]]
        if len(acts) != n_steps or any(len(a) != 6 for a in acts):
            bad.append("the random AgentOrders are not one [active, side, exec, p, q, e] per step")
        for k, (act, sd, ex, pp, qq, ee) in enumerate(a for a in acts if len(a) == 6):
            lo_q = 2 if ex == ICE else 1
            if act not in (0, 1) or sd not in (BUY, SELL) or ex not in (NON, ICE, IOC, FOK) \
                    or not mid - H <= pp <= mid + H or not lo_q <= qq <= Q_AGENT_MAX \
                    or not ((1 <= ee <= qq - 1) if ex == ICE else ee == 0):
                bad.append(f"random AgentOrder of step {k + 1} outside the step-5 distribution: {acts[k]}")
            elif act and (int(encode_price(float(np.float32((pp - O_TICKS) * PRICE_TICK_EUR)), CONFIG)) != pp
                          or int(encode_qty(float(np.float32(qq * QTY_TICK_MW)), CONFIG)) != qq):
                bad.append(f"random AgentOrder of step {k + 1}: its EUR / MW values do not encode back to {pp} / {qq}")
        # the twins, rebuilt from the params, byte for byte
        tw = p["twins"]
        if set(draw.aux_tapes) != {"B", "CTRL"} | {f"B@{a}" for a in parts}:
            bad.append(f"aux tapes {sorted(draw.aux_tapes)}")
        for name, x in tw.items():
            keep = x["keep"]
            sig = {int(k): int(v) for k, v in x["sigma"].items()}
            tau = {int(k): int(v) for k, v in x["tau"].items()}
            if sorted(sig) != sorted(parts) or sorted(sig.values()) != sorted(parts):
                bad.append(f"{name}: sigma is not a permutation of the participants")
            fixed_ag = [a for a in parts if sig[a] == a]
            if fixed_ag != ([] if keep is None else [int(keep)]):
                bad.append(f"{name}: sigma fixes {fixed_ag} (expected {[] if keep is None else [keep]})")
            foreign = [int(o["oid"]) for o in ords if keep is None or int(o["agent"]) != int(keep)]
            if sorted(tau) != sorted(foreign) or sorted(tau.values()) != sorted(foreign) \
                    or any(tau[o] == o for o in foreign):
                bad.append(f"{name}: tau is not a derangement of the foreign order ids")
            want = _tape(twin_orders(ords, keep, x["sigma"], x["tau"]))
            if not np.array_equal(want, np.asarray(draw.aux_tapes.get(name, np.zeros((0, 13))))):
                bad.append(f"{name}: aux tape differs from its rule")
        c = p["ctrl"]
        ctrl = twin_orders(ords, None, tw["B"]["sigma"], tw["B"]["tau"])
        ctrl[int(c["row"])]["qty_int"] += 1
        if not np.array_equal(_tape(ctrl), np.asarray(draw.aux_tapes.get("CTRL", np.zeros((0, 13))))) \
                or int(ords[int(c["row"])]["exec"]) != ICE or int(ords[int(c["row"])]["oid"]) != int(c["oid_A"]):
            bad.append("CTRL is not B plus one tick on an iceberg's peak")
        # the walks: A accepted, B and every B@a are A relabelled (§6.6, §5.2), and every B@a keeps a's icebergs
        wA = walk(t)
        att = {"orders": ords, "participants": parts, "n_steps": n_steps}
        why = _reject_reason(att, wA)
        if why is not None:
            bad.append(f"world A fails the acceptance test: {why}")
        ts = touch_steps(wA, len(ords), n_steps)
        if not ts or p["touch_steps"] != ts or [int(c["step"]), int(c["oid_A"])] != [ts[0][0], ts[0][1][0]]:
            bad.append("touch steps / control step disagree with the walk")
        for name, x in tw.items():
            w = walk(draw.aux_tapes[name])
            if w["trouble"] or w["trades"] != relabel(wA["trades"], x["sigma"], x["tau"]):
                bad.append(f"{name}: its walk is not A's relabelled")
            keep = x["keep"]
            want_ice = sorted(j for j, v in wA["ice"].items() if keep is not None and v["owner"] == int(keep))
            if sorted(w["ice"]) != want_ice:
                bad.append(f"{name}: icebergs {sorted(w['ice'])} kept, expected {want_ice} (side-condition)")
    except Exception as exc:  # noqa: BLE001
        bad.append(f"params unreadable: {type(exc).__name__}: {exc}")
    return bad


# ---- the expectation on the first draw, and the classes of the initial range ------------------------------------
if FIRST is not None:
    _d = DRAWN[FIRST]
    _e = expectation(_d)
    print(f"seed {FIRST} classes: {_e['classes']}")
    print(f"  A trades [buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission]: "
          f"{_e['A']['trades']}")
    print(f"  B trades (= A relabelled): {_e['B']['trades']}")
    print(f"  A hidden remaining: {_e['A']['hidden_remaining']}; B hidden remaining: {_e['B']['hidden_remaining']}")
    print(f"  reports (legs [price_int, qty_int], no side: correction J4): {_e['reports']}")
    print(f"  ownership: {_e['ownership']}")
    print(f"  wrong reports discriminated: {_e['disc']}")
    print(f"  precondition departures: {precondition(_d)}")
    _j0, _en0 = next(iter(_e["ice_entry"].items()))
    _fm0 = Q_AGENT_MAX * _e["n_steps"]
    print(f"  (2r) illustration only (f is read from the run): iceberg {_j0} {_en0}; had its entry swept f = 1 / "
          f"f = f_max = {_fm0} ticks of resting participant orders: {surfacing(_en0, 1)} / {surfacing(_en0, _fm0)}")
# ---- (2r) analytically, over the initial seed range (K2 r4, findings F17 and F19) ----------------------------------
# For every iceberg of every built draw and every entry sweep f the participant could supply (1 <= f <= f_max), the
# amended item (4) must give k_e = 0, u = f, a zero visible gap and the hidden remainder h. The retired display C1
# would give a gap of f instead: the count of (iceberg, f) pairs where C1 and C3 disagree.
SURF_SCAN = Counter()
SURF_BAD = []
for _s, _d in sorted(DRAWN.items()):
    if _d is None or _s >= SEED0 + DRAWS:
        continue
    _ex = expectation(_d)
    _fm = Q_AGENT_MAX * _ex["n_steps"]
    for _j, _en in _ex["ice_entry"].items():
        SURF_SCAN["icebergs"] += 1
        if not _fm < _en["peak"]:
            SURF_BAD.append([_s, _j, "f_max >= peak"])
        for _f in range(1, _fm + 1):
            _v = surfacing(_en, _f)
            SURF_SCAN["(iceberg, f) pairs"] += 1
            if not (_v["k_e"] == 0 and _v["u"] == _f and _v["visible_gap"] == 0 and _v["hidden_unchanged"]):
                SURF_BAD.append([_s, _j, _f, _v])
            if _v["visible_gap"] != 0:
                SURF_SCAN["pairs with a non-zero rule-implied gap (C3)"] += 1
            if _v["retired_C1"]["visible_gap"] != _v["visible_gap"]:
                SURF_SCAN["pairs where the retired C1 display disagrees (u > 0)"] += 1
print(f"(2r) scan over the initial seed range (amended item (4), display C3): {dict(SURF_SCAN)}; "
      f"departures from k_e = 0, u = f, gap 0, hidden h: {len(SURF_BAD)}" + (f" {SURF_BAD}" if SURF_BAD else ""))
_init_classes = Counter(c for d in DRAWN.values() if d is not None for c in expectation(d)["classes"])
display(Markdown("**Classes in the initial seed range** (draws containing each)\n\n| class | draws |\n|---|---|\n"
                 + "\n".join(f"| {c} | {_init_classes.get(c, 0)} |" for c in ALL_CLASSES)))

## 6. Run

### The engine, the trade reports and the environment, eager and jitted

> **Wired at K3 (r2).** The developer wires this section at K3 and edits **nothing else** (independence rule 2): sections 1–5 and 7 are the expectation. A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **four names** from this section, plus one optional name. While any of the four is not defined, section 7 records a failing `section 6 wired` check, runs no draw, and the notebook ends `NOT CERTIFIED`. `draw` is always a `Draw` built by section 4's `make_draw(seed)`; `world_tape(draw, world)` gives the tape of a world. Every function may raise; section 7 records a raise as a failed check. Every call starts from its own fresh book (or fresh episode).

1. **`run_draw(draw, world, *, jit) -> RunRecord`**, `world` in `("A", "B")`: the battery's record builder on that world's tape, i.e. `tests.omiecid.reference.omie_invariants.build_record(world_tape(draw, world), capacity_bounds=synth.tape_capacity_bounds(world_tape(draw, world)), jit=jit)`, with no other argument (one fresh book of an OPEN contract, default `OMIEProductConfig()` / `OMIEMarketConfig()`, no cancellation, no gate expiry). `jit=False` runs eager.

2. **`agent_report(draw, world, agent_id, *, jit) -> list`**, `world` in `("A",)` plus every `B@<a>` of `observer_worlds(draw)`: the card's `get_agent_trades(a)`, i.e. what **`ContractBook.get_agent_trades(state, agent_id)`** returns for the final book state after replaying `world_tape(draw, world)`'s real rows, in order, on one fresh book of an OPEN contract through the production submission path (the path the record builder drives), with no `Session` participant. Return it **unchanged in content and order**, each element converted to a plain Python list of its scalar values (`np.asarray(row).ravel().tolist()`). Nothing may be added, removed, reordered, redacted or decoded. `jit` selects eager (`jax.disable_jit()`) or jitted for the replay.

3. **`report_leg(row, agent_id) -> [price_int, qty_int]`**: reads one row returned by `agent_report` for `agent_id` and returns that trade's price in price ticks and its quantity in quantity ticks, read from the row's own fields only (never from the tape, the walk or any other run). No side (correction J4, finding F16). Section 7 uses it only for the partition (5.4); the anonymity twin (5.5) compares the raw rows.

4. **`run_env(draw, world, policy, *, jit) -> dict`**, `world` in `("A", "B", "CTRL")`, `policy` in `("noop", "random")`: one episode of `omiecid_simulator.omiecid.omie_session.Session` built with exactly `ENV_CONFIG` (section 3) plus the tapes, replaying `world_tape(draw, world)`. *(REL-09)* The participant submits `AgentOrder.none()` on every step for `"noop"`, and `agent_order(draw.params["actions"][k])` (section 4, step 5) on step `k` for `"random"`. The episode is `reset` followed by exactly `draw.params["n_steps"]` calls of `step_env` (whatever the done flag says; after `done` the session ignores the call. Early-done ruling, 2026-10-26: the default span-scaled clock can fire `done` before the last real slice; that is harmless to the twin because every world of a draw carries the same timestamp column, checked in the precondition, and the policy does not move the clock, so all worlds and both policies freeze at the same step), all with the same PRNG key in every world. **The two (or three) worlds of a draw must run under the same `Session` instance and configuration, from a fresh reset, differing only in which tape is replayed** (a `Session` is a static jit key that includes its tape set, so one instance over a stacked tape set — `synth.stack_tapes` — of every world of every seed the loop may need, with the episode's tape index pinned after `reset`, avoids one compilation per draw; how the index is pinned is section 6's choice). `jit=False` runs the whole episode under `jax.disable_jit()`. The dict has exactly these keys:
   - `"tob"`: `Session.top_of_book(state)` after `reset` and after each step (`n_steps + 1` entries), each a `{field: np.ndarray}` of its fields exactly as returned (nothing rescaled or cast);
   - `"lob"`: `ContractBook.get_anonymous_lob(state.book)` at the same instants (`n_steps + 1` entries), each `{"bids": (n, 2) float64, "asks": (n, 2) float64}` of the returned `(price, quantity)` levels, in returned order;
   - `"fills"`, `"dones"`: the `FillReport` (a `{field path: np.ndarray}` of its leaves, `submit` included) and the done flag returned by each step (`n_steps` entries each, `np.asarray`);
   - `"own_report"`: `ContractBook.get_agent_trades(final_book_state, AGENT_TRADER_ID)` on the episode's final book, as in item 2 (`AGENT_TRADER_ID` from `omie_session`);
   - `"own_orders"`: `ContractBook.get_agent_orders(final_book_state, AGENT_TRADER_ID)`, each returned order as a plain list of its field values, in returned order;
   - `"cash"`, `"inventory"`: the final `state.agent_cash` and `state.agent_inventory` (`np.asarray`);
   - `"book_final"`: the episode's final book as a `tests.omiecid.reference.omie_invariants.BookSnapshot` (section 7 reads `hidden_qty`, `visible_qty`, `resting`, `book_full`, `truncated`, `trade_rows_used` and the leaves `bids`, `asks`, `trades`, `iceberg_meta`, `n_active`, `ts_counter`);
   - `"agent_trades"`: the final session state's per-agent trade tape (`state.agent_trades`) as an int array (reported only, finding F13).

5. **`own_orders_view(draw, world, agent_id, *, jit) -> dict`** (wired at K3 from `ContractBook.get_agent_orders`; optional in the contract): only if the simulator exposes an **owner-facing** view of an agent's own active orders through a public API (not the harness's engine-state reader): `{order_id: full remaining quantity in ticks}` as that view shows them to `agent_id`, after replaying `world_tape(draw, "A")` as in item 2. If no such view exists, leave the name undefined; section 7 records that, and asserts nothing about it (finding F6).

Section 7 reads these `RunRecord` fields: `accepted` (its keys), `rejections` (each `oid`), `fills` (each `submission`, `row_index`, `buyer_oid`, `seller_oid`, `buyer_agent`, `seller_agent`, `price_int`, `qty_int`), `filled`, `still_resting`, `hidden_remaining`, `evicted`, `cancelled_or_expired` (each `cause`, `qty_int`), `final_snapshot` (its `resting(side)`), `observed_capacity`, and the whole record through `run_battery` and `compare_records`.

Section 7 calls, on every draw it uses: `run_draw(draw, "A", jit=True)`, `run_draw(draw, "B", jit=True)`; `agent_report(draw, "A", a, jit=True)` for every id of the partition and `agent_report(draw, f"B@{a}", a, jit=True)` for every participant; `run_env(draw, w, pol, jit=True)` for `(A, noop)`, `(B, noop)`, `(CTRL, noop)`, `(A, random)`, `(B, random)`; `own_orders_view(draw, "A", a, jit=True)` if defined. On the eager selection of section 3 it repeats the `A` engine run, the `A` reports and the four `A`/`B` episodes with `jit=False`. A seed beyond the initial range (a re-draw after a discard) is built with `prepare(seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_draw, agent_report, report_leg, run_env and own_orders_view, per the
# contract above.
#
# run_draw: the battery's record builder on the world's tape, with its analytic bounds and
# no other argument (fresh OPEN book, default configs, no cancels, no gate expiry).
#
# agent_report: the record builder's submission path (ContractBook.submit_order on the tape
# row decoded by _row_to_order_traced), real rows in order, on one fresh OPEN book of the
# record builder's contract; then ContractBook.get_agent_trades on the final state, each
# element converted with np.asarray(row).ravel().tolist() and nothing else.
#
# report_leg: the row get_agent_trades returns is (price_eur, qty_mw, sec). Price and
# quantity decode back to ticks with the engine's encoders (encode_price / encode_qty of
# the default OMIEProductConfig); the leg is [price_int, qty_int], no side (J4).
#
# run_env (REL-09: the Session participant view): ONE Session built with exactly ENV_CONFIG plus the tapes, over a stacked tape set
# (synth.stack_tapes) holding worlds A, B and CTRL of every seed the run loop may visit
# (SEED0 .. SEED0 + MAX_ATTEMPTS - 1, built here with section 4's prepare(); the loop then
# reuses them). The env is a static jit key that includes its tape set, so one instance
# means one compilation of reset / step_env. After reset (the same PRNG key in every world)
# the episode's tape_idx is pinned to the world's row of the stacked set; the participant view
# (top_of_book, get_anonymous_lob) is read after reset and after each step. Then exactly n_steps
# calls of step_env with the same key in every world (AgentOrder.none() for noop, the recorded
# random AgentOrder for random), each returning (state, FillReport, done).
#
# own_orders_view: ContractBook.get_agent_orders -- the simulator's public owner-facing view
# of an agent's own resting orders (an iceberg at visible + hidden, S6) -- on the final
# state of the A replay, read as {order_id: quantity in ticks as the view shows it}. An
# order id shown twice raises (the view must list each own order once).
# ---------------------------------------------------------------------------------------
import contextlib

import jax
import jax.numpy as jnp

from omiecid_simulator.jaxomiecid.omie_data_loader import _row_to_order_traced
from omiecid_simulator.omiecid.contract import Contract, ContractStatus
from omiecid_simulator.omiecid.contract_book import ContractBook
from omiecid_simulator.omiecid.encoding import encode_price, encode_qty
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from omiecid_simulator.omiecid.omie_session import Session
from tests.omiecid.reference import omie_invariants as _inv
from tests.omiecid.reference.omie_invariants import build_record

ENV_WORLDS = ("A", "B", "CTRL")
REPORT_CONTRACT = Contract(contract_id=0, delivery_start_minutes=_inv._DELIVERY_START,
                           delivery_end_minutes=_inv._DELIVERY_END, gct_minutes=_inv._GCT,
                           status=ContractStatus.OPEN)   # the record builder's contract (every row on contract 0)
REPORT_BOOK = ContractBook(REPORT_CONTRACT, OMIEProductConfig(), OMIEMarketConfig())
ENV_KEY = jax.random.PRNGKey(0)   # the same reset / step key in every world


def _mode(jit):
    return contextlib.nullcontext() if jit else jax.disable_jit()


def run_draw(draw, world, *, jit):
    """The battery's record builder on `world`'s tape (A or B), nothing else."""
    if world not in ("A", "B"):
        raise ValueError(f"run_draw: world {world!r}")
    t = world_tape(draw, world)
    return build_record(t, capacity_bounds=synth.tape_capacity_bounds(t), jit=jit)


def _replay_state(tape, jit):
    """(book, final state) after the tape's real rows, in order, on one fresh OPEN book."""
    rows = real_rows(tape)
    if len({int(r[C_CONTRACT]) for r in rows}) > 1:
        raise ValueError("mixed contract ids on one tape")
    book = ContractBook(REPORT_CONTRACT, OMIEProductConfig(), OMIEMarketConfig())
    with _mode(jit):
        state = book.init()
        for r in rows:
            state, _res = book.submit_order(state, _row_to_order_traced(jnp.asarray(r), book._product_config))
    return book, state


def agent_report(draw, world, agent_id, *, jit):
    """ContractBook.get_agent_trades(final state, agent_id), unchanged, each row as a plain list."""
    if world != "A" and world not in observer_worlds(draw).values():
        raise ValueError(f"agent_report: world {world!r} is not A or a B@<a> of this draw")
    book, state = _replay_state(world_tape(draw, world), jit)
    with _mode(jit):
        rows = book.get_agent_trades(state, int(agent_id))
    return [np.asarray(r).ravel().tolist() for r in rows]


def report_leg(row, agent_id):
    """[price_int, qty_int] of one get_agent_trades row, from the row's own fields only (no side: J4)."""
    vals = list(row)
    if len(vals) != 3:
        raise ValueError(f"unexpected report row layout {vals!r}")
    price_eur, qty_mw, _sec = vals
    return [int(encode_price(float(price_eur), CONFIG)), int(encode_qty(float(qty_mw), CONFIG))]


def own_orders_view(draw, world, agent_id, *, jit):
    """ContractBook.get_agent_orders(final A state, agent_id) as {order_id: quantity ticks shown}."""
    book, state = _replay_state(world_tape(draw, "A"), jit)
    with _mode(jit):
        orders = book.get_agent_orders(state, int(agent_id))
    out = {}
    for o in orders:
        oid = int(np.asarray(o.order_id))
        if oid in out:
            raise ValueError(f"get_agent_orders shows order {oid} of agent {int(agent_id)} more than once")
        out[oid] = int(encode_qty(float(np.asarray(o.quantity)), CONFIG))
    return out


# ---- one env over the stacked tapes of every world of every seed the loop may visit -------------------------------
for _s in range(SEED0, SEED0 + MAX_ATTEMPTS):
    if _s not in DRAWN:
        prepare(_s)
ENV_INDEX, _env_tapes = {}, []
for _s in range(SEED0, SEED0 + MAX_ATTEMPTS):
    _d = DRAWN.get(_s)
    if _d is None:
        continue
    for _w in ENV_WORLDS:
        ENV_INDEX[(int(_d.seed), _w)] = len(_env_tapes)
        _env_tapes.append(np.asarray(world_tape(_d, _w)))
ENV_TAPES = synth.stack_tapes(_env_tapes)
ENV = Session({**ENV_CONFIG, "tapes": ENV_TAPES})


def _tape_index(draw, world):
    i = ENV_INDEX.get((int(draw.seed), world))
    t = np.asarray(world_tape(draw, world))
    if i is None or not np.array_equal(ENV_TAPES[i][: t.shape[0]], t) \
            or np.any(ENV_TAPES[i][t.shape[0]:, C_OID] >= 1):
        raise KeyError(f"seed {draw.seed} world {world}: not in the stacked env tape set")
    return i


def _leaves(tree):
    # A NamedTuple pytree (TopOfBook, FillReport) as {field path: np.ndarray}, exactly as returned.
    return {jax.tree_util.keystr(path): np.asarray(x) for path, x in jax.tree_util.tree_flatten_with_path(tree)[0]}


def _lob(book_state):
    # get_anonymous_lob's (bids, asks) levels, each as an (n, 2) float64 array of (price, quantity) as returned.
    bids, asks = REPORT_BOOK.get_anonymous_lob(book_state)
    return {"bids": np.asarray(bids, dtype=np.float64).reshape(-1, 2),
            "asks": np.asarray(asks, dtype=np.float64).reshape(-1, 2)}


def run_env(draw, world, policy, *, jit):
    """One Session episode replaying `world`'s tape: reset, then exactly n_steps step_env calls."""
    if world not in ENV_WORLDS or policy not in POLICIES:
        raise ValueError(f"run_env: world {world!r} / policy {policy!r}")
    idx = _tape_index(draw, world)
    n = int(draw.params["n_steps"])
    orders = [AgentOrder.none()] * n if policy == "noop" else [agent_order(a) for a in draw.params["actions"]]
    tob, lob, fills, dones = [], [], [], []
    with _mode(jit):
        state = ENV.reset(ENV_KEY)
        state = state.replace(tape_idx=jnp.int32(idx))
        tob.append(_leaves(ENV.top_of_book(state)))
        lob.append(_lob(state.book))
        for k in range(n):
            state, rep, d = ENV.step_env(ENV_KEY, state, orders[k])
            tob.append(_leaves(ENV.top_of_book(state)))
            lob.append(_lob(state.book))
            fills.append(_leaves(rep))
            dones.append(np.asarray(d))
        own = REPORT_BOOK.get_agent_trades(state.book, PARTICIPANT_ID)
        own_orders = REPORT_BOOK.get_agent_orders(state.book, PARTICIPANT_ID)
    return {"tob": tob, "lob": lob, "fills": fills, "dones": dones,
            "own_report": [np.asarray(x).ravel().tolist() for x in own],
            "own_orders": [[np.asarray(v).item() for v in o] for o in own_orders],
            "cash": np.asarray(state.agent_cash), "inventory": np.asarray(state.agent_inventory),
            "book_final": _inv._snapshot(state.book),
            "agent_trades": np.asarray(state.agent_trades)}


print(f"run_env: one Session over {ENV_TAPES.shape[0]} stacked tapes (worlds {ENV_WORLDS} of "
      f"{len(ENV_INDEX) // len(ENV_WORLDS)} seeds, shape {ENV_TAPES.shape}), config {ENV_CONFIG} + tapes; "
      f"steps_per_episode {ENV.steps_per_episode}, tape_seconds_span {ENV.tape_seconds_span}")

## 7. Certification

The oracle comparisons come first, eight assertions per valid draw (section 5, items 1–7 and 6″); then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw.** Each assertion's `expected` and `observed` are canonical compact JSON (sorted keys), **in full** and never truncated. The participant view appears as one blake2b digest per step each of `top_of_book`, `get_anonymous_lob`, `FillReport` and `done` (field names, dtype, shape and bytes); the end-of-episode own trades, own orders, cash and inventory are recorded in full; when the twins differ the `detail` lists **every** differing step and field with both values. (1) `engine`: section 5's table for `A` and `B` against the two records. (2) `participant-view twin (noop)` and (2′) `participant-view twin (random)`: world `A`'s per-step view against world `B`'s, **where the premise of F17 holds**; where it fails (a tape iceberg's entry swept the participant's resting order), (2r) `rule-implied display` instead: the entry fills in `A` and `B` ($1 \le f_j \le f_{\max} < P_j$, $f_{\max}$ = `Q_AGENT_MAX` $\times K$, participant passive), the rule-implied values of the amended item (4) ($k_e = 0$, $u_j = f_j$, $v_1 = P_j - f_j$) giving a visible gap of $0$, the hidden $h_j$ and one resting row at $b_j$ on the episode's final book, and the whole participant view bit-identical, with the per-step digests of both worlds recorded in full (K2 r3, re-derived at K2 r4). (3) `positive control`: `CTRL`'s noop `top_of_book` must differ from `B`'s on some step, and `get_anonymous_lob` on that step as well. (4) `trade-report partition`: the expected legs of every id and their union against `report_leg` of the `A` reports (`report_leg(row, a)` returns `[price_int, qty_int]`, no side: J4), and the row count. (5) `trade-report twin`: every participant's raw `A` report against its raw `B@a` report. (6) `ownership`: each iceberg's resting row (price, owner, visible) and hidden remainder in the `A` record (and, if `own_orders_view` is defined, the separate assertion (6′) on it, expected to pass: S6). (6″) `ownership (participant)`: in the `random` episodes, `get_agent_orders(final_book, AGENT_TRADER_ID)` against the participant's resting rows of the final book (visible + hidden), its bit-identity between `A` and `B`, and the anonymous book's quantity at each price where a participant iceberg rests with a hidden part (the visible sum, F23). (7) `the twin stayed a twin`: each tape iceberg's hidden remainder at the end of the two `A` episodes ($h_j$, an iceberg of (2r) included: its rule-implied $R_j - v_1$ is $h_j$), none of a tape order in `B`/`CTRL`, and no capacity flag in any episode; the participant's own icebergs are excluded (F4). The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**Aggregate assertions:** the section-6 wiring; the dataset precondition (section 4), listing every departing seed; the **discrimination** of the three wrong reports on every valid draw; the battery on both engine worlds — **RB, I1–I7 and I9 never fail**: each is `pass` or `not_applicable`, and `not_applicable` counts as **vacuously satisfied**, never as "not pass" (carry-forward rule, finding F9; the counts are reported); **no capacity flag inside the margin** on either engine world (finding F10); the hop bound ≤ 160, trades ≤ 80 and ≤ 64 slices per iceberg on every valid draw, the env-side analytic bounds included; **coverage** — every class of section 3 among the valid draws; **non-vacuity of the participant's own report** — non-empty in the random policy on at least one valid draw; **non-vacuity of display C3** — at least one valid draw whose jitted `random` episode fails the premise with $u > 0$ (K2 r4, finding F19); **non-vacuity of (6″)** — at least one valid draw whose `random` episode ends with a participant iceberg resting with a hidden part; **non-vacuity of both sides** — at least one valid draw on which the participant trades as buyer, and at least one on which it trades as seller; the eager comparisons — at least one in all and one per eager class, a missing one recorded as a failure: integer fields equal (records through `compare_records`, reports, done flags, the participant's own trades and own orders, the anonymous book, the integer fields of `top_of_book` and the `FillReport`, the final session book), and the participant-view twin, (2) or (2r), holding in eager mode too.

The battery runs on both jitted engine records with `expects_trade=True`. **Reported, not asserted:** the premise check's counts per policy and every premise-failed draw in full (seed, policy, icebergs, $f_j$, $u_j$; finding F17, never a silent discard), the valid draws per class, the discards per reason, the eager seeds, the battery status counts, the largest eager-vs-jitted difference of the float fields of `top_of_book`, the `FillReport`, cash and inventory, whether `state.agent_trades` was bit-identical across the twins (F13), whether an owner-facing own-orders view exists (F6), the size of the participant's own reports, the count of participant orders by execution restriction and side, the participant FOK orders filled and killed, participant self-matches (F8), the participant iceberg peaks exhausted, and the absence of LOB timestamps (F24).

In [ ]:
import traceback
from collections import defaultdict

from omiecid_simulator.omiecid.omie_session import AGENT_ORDER_ID_BASE
from omiecid_simulator.omiecid.order_types import OMIEOrder
from tests.omiecid.reference.omie_invariants import _fill_from_row, compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Anonymity and information visibility",
    known_result="The Session participant view (top_of_book and get_anonymous_lob after reset and every step, the "
                 "FillReport and done flag of every step, own trades, own orders, cash and inventory at the end) is "
                 "bit-identical between a world "
                 "with foreign resting icebergs and its twin with every foreign hidden remainder removed and every "
                 "identifier permuted (noop and random AgentOrder policy) where no tape iceberg fills a resting "
                 "participant order on "
                 "entry, and where one does, the rule-implied display of section 4.2.1 Rule 2 (1), (4) as amended "
                 "2026-10-20 (remainder as instance k_e = floor(M/P) showing v1 = min(P - u, R); here k_e = 0, u = f, so "
                 "A shows P - f like B, hidden h, and the twin stays bit-identical on every step), while a one-tick "
                 "change of the visible touch is seen by top_of_book; the participant's own-orders reader shows its "
                 "own icebergs at full quantity, the anonymous book only their visible slice; "
                 "get_agent_trades(a) returns exactly a's legs [price_int, qty_int], byte-identical when every identifier but a's is "
                 "permuted, and the union over agents is every trade once per side; each iceberg rests under its "
                 "owner with its full remaining quantity held; both engine worlds' trades pinned by the walk",
    oracle_kind="exact-parametric, carried by a metamorphic twin", quotes=QUOTES, citations=CITATIONS,
    constructions=(), requested_draws=DRAWS, runtime_forecast_s=180.0, started_at=T0,
    downstream="every result obtained through the Session participant view (strategies, studies, learned policies); C08-C10 "
               "(hidden quantity once it legitimately surfaces); C15 (M6 ownership blindness at scale)",
)

R_ENGINE = "§6.1; §6.2 items 2-6; §4.5; §4.2; §6.6; §5.2; review D3-08 (counterparties pinned)"
R_OBS = "§7.1; §7.3; §9.2 'LOB per contract', 'Best bid / best ask'; review §2 (the metamorphic formulation)"
R_OWNP = "§9.2 'Own orders' (showing full iceberg quantity) and 'LOB per contract' (aggregated visible quantities); findings F6, F23"
R_CTRL = "rule 1 (every check seen failing): a one-tick change of the visible touch is seen by top_of_book (§9.2 'Best bid / best ask', visible quantities)"
R_PART = "§7.2 (own trades only; notified to the agent in question); §9.2 'Trade book'"
R_ANON = "§7.2 'Counterparty identity is not disclosed'; §7.1 (own operations may be identified); finding F7"
R_OWN = "§9.2 'Own orders' (showing full iceberg quantity); §7.3 identity kept internally; finding F6"
R_TWIN = "§4.2 (a regenerated slice legitimately shows hidden quantity): the twin is execution-equivalent (F4)"
R_BAT = "the battery (RB, I1-I9) on both engine worlds of every valid draw"
R_CAP = "capacity-validity precondition (five ceilings), env-side margin included (F10)"
R_DATA = "dataset precondition (section 4); §2.3"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"
R_DISC = "rule 1 (every check seen failing): each named wrong report misses the rule for some participant"
R_SURF = ("§4.2.1 Rule 2, Incoming iceberg (1) total quantity, (2) instance chain, (4) remainder as instance k_e "
          "showing v1 = min(P - u, R), (6) instance limits, amended 2026-10-20; OQ-34 display C3; OMIE Rules Annex "
          "§4.4 :10256-10261; anonymity concerns identity (§7.1, §7.3); findings F17, F19 (K2 r3, r4)")

MUST_PASS = ("RB", "I2", "I3", "I6", "I7", "I9")   # expected 'pass' on every C13 engine world
NO_FAIL = ("I1", "I4", "I5")                        # may be not_applicable (F9)
SATISFIED = ("pass", "not_applicable")              # carry-forward rule: not_applicable is vacuously satisfied


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a value", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_ENGINE)
        return None


def nonzero(m):
    return {str(int(k)): int(v) for k, v in sorted(m.items(), key=lambda kv: int(kv[0])) if int(v)}


def plain(rows):
    # A report as returned, each row a plain list of its scalars.
    return [np.asarray(r).ravel().tolist() for r in rows]


def digest(x):
    # Bit identity of one returned value: dtype, shape and bytes.
    a = np.ascontiguousarray(np.asarray(x))
    h = hashlib.blake2b(digest_size=16)
    h.update(f"{a.dtype.str}|{a.shape}|".encode("ascii"))
    h.update(a.tobytes())
    return h.hexdigest()


def observed_trades(rec):
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    return [[int(f.buyer_oid), int(f.seller_oid), int(f.buyer_agent), int(f.seller_agent), int(f.price_int),
             int(f.qty_int), int(f.submission)] for f in fills]


def observe_engine(rec):
    # The observed counterpart of section 5 (1) for one world.
    return {
        "accepted": sorted(int(a) for a in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "trades": observed_trades(rec),
        "filled": nonzero(rec.filled),
        "still_resting": nonzero(rec.still_resting),
        "hidden_remaining": nonzero(rec.hidden_remaining),
        "cancelled": {str(int(q)): [[str(c.cause), int(c.qty_int)] for c in cs]
                      for q, cs in sorted(rec.cancelled_or_expired.items(), key=lambda kv: int(kv[0])) if cs},
        "evicted": nonzero(rec.evicted),
    }


def observe_ownership(rec, ice_oids):
    # Section 5 (6), read from the engine's final book and hidden remainders.
    rows = list(rec.final_snapshot.resting(BUY)) + list(rec.final_snapshot.resting(SELL))
    out = {}
    for j in ice_oids:
        mine = [r for r in rows if int(r[2]) == int(j)]
        vis = sum(int(r[1]) for r in mine)
        hid = int(rec.hidden_remaining.get(int(j), 0))
        one = len(mine) == 1
        out[str(j)] = {"owner": int(mine[0][3]) if one else [int(r[3]) for r in mine],
                       "price_int": int(mine[0][0]) if one else [int(r[0]) for r in mine],
                       "visible": vis, "hidden": hid, "full_remaining": vis + hid}
    return out


def tree_digest(leaves):
    # Bit identity of one reader's return value ({field: array}): field names, dtypes, shapes and bytes.
    h = hashlib.blake2b(digest_size=16)
    for k in sorted(leaves):
        a = np.ascontiguousarray(np.asarray(leaves[k]))
        h.update(f"{k}|{a.dtype.str}|{a.shape}|".encode("utf-8"))
        h.update(a.tobytes())
    return h.hexdigest()


def env_view(tr):
    # The participant view of one episode: one digest per step of top_of_book, get_anonymous_lob, FillReport and
    # done; the end-of-episode own trades and own orders in full, cash and inventory in full and as digests.
    return {"n_tob": len(tr["tob"]), "tob": [tree_digest(x) for x in tr["tob"]],
            "lob": [tree_digest(x) for x in tr["lob"]],
            "n_steps": len(tr["fills"]), "fills": [tree_digest(x) for x in tr["fills"]],
            "dones": [digest(d) for d in tr["dones"]], "own_report": plain(tr["own_report"]),
            "own_orders": tr["own_orders"], "cash": digest(tr["cash"]), "inventory": digest(tr["inventory"]),
            "cash_inventory": [np.asarray(tr["cash"]).tolist(), np.asarray(tr["inventory"]).tolist()]}


def _leaf_diff(out, field, k, xa, xb, la, lb):
    # Every differing leaf element of one reader value ({name: array}) at step k, both values in full.
    for name in sorted(set(xa) | set(xb)):
        if name not in xa or name not in xb:
            out.append({"field": field, "step": k, "leaf": name, "present": [name in xa, name in xb]})
            continue
        x, y = np.asarray(xa[name]), np.asarray(xb[name])
        if x.dtype != y.dtype or x.shape != y.shape:
            out.append({"field": field, "step": k, "leaf": name, la: f"{x.dtype}{x.shape}", lb: f"{y.dtype}{y.shape}",
                        "values": [x.tolist(), y.tolist()]})
            continue
        xf, yf = x.ravel(), y.ravel()
        for i in range(xf.size):
            if xf[i:i + 1].tobytes() != yf[i:i + 1].tobytes():
                out.append({"field": field, "step": k, "leaf": name, "index": i, la: repr(xf[i].item()),
                            lb: repr(yf[i].item())})


def env_diff(ta, tb, la="A", lb="B"):
    # Every differing step and field of the participant view, with both values in full.
    out = []
    for key in ("tob", "lob", "fills", "dones"):
        xa, xb = ta[key], tb[key]
        if len(xa) != len(xb):
            out.append({"field": key, "lengths": [len(xa), len(xb)]})
        for k, (x, y) in enumerate(zip(xa, xb)):
            if key == "dones":
                x, y = {"done": x}, {"done": y}
            _leaf_diff(out, key, k, x, y, la, lb)
    for key in ("cash", "inventory"):
        _leaf_diff(out, key, None, {key: ta[key]}, {key: tb[key]}, la, lb)
    if plain(ta["own_report"]) != plain(tb["own_report"]):
        out.append({"field": "own_report", la: plain(ta["own_report"]), lb: plain(tb["own_report"])})
    if ta["own_orders"] != tb["own_orders"]:
        out.append({"field": "own_orders", la: ta["own_orders"], lb: tb["own_orders"]})
    return out


def env_hidden(tr, oids):
    return {str(j): int(tr["book_final"].hidden_qty(int(j))) for j in oids}


def env_flags(tr):
    s = tr["book_final"]
    return {"book_full": bool(s.book_full), "truncated": bool(s.truncated), "trade_rows_used": int(s.trade_rows_used())}


def snap_key(s):
    return [digest(getattr(s, k)) for k in ("bids", "asks", "trades", "iceberg_meta")] + \
        [int(s.n_active), int(s.ts_counter), bool(s.book_full), bool(s.truncated)]


def env_bounds(tape, n_steps):
    # Finding F10: the tape's analytic bounds plus the participant: per step at most one order of <= Q_AGENT_MAX
    # ticks (so <= Q_AGENT_MAX trades, one resting row, one iceberg slot, and <= Q_AGENT_MAX slices of its own).
    cb = synth.tape_capacity_bounds(tape)
    units = sum(ceil_div(r[C_QTY], r[C_PEAK]) if int(r[C_EXEC]) == ICE else 1 for r in real_rows(tape))
    return {"resting_per_side": int(cb["resting_per_side"]) + n_steps,
            "trades": int(cb["trades"]) + Q_AGENT_MAX * n_steps,
            "hops": max(int(cb["hops"]) + Q_AGENT_MAX * n_steps, units + Q_AGENT_MAX * n_steps + Q_AGENT_MAX + 2),
            "concurrent_icebergs": int(cb["concurrent_icebergs"]) + n_steps}


LIM = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
       "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
PEAK = {"hop_bound": 0, "env_hop_bound": 0, "trades": 0, "slices": 0, "env_trade_rows": 0}


def margin_breaches(draw):
    # The capacity-validity precondition (five ceilings), checked from the tapes before they are run.
    bad = []
    n = int(draw.params["n_steps"])
    worlds = ["A", "B", "CTRL"] + sorted(observer_worlds(draw).values())
    for w in worlds:
        t = world_tape(draw, w)
        bd = synth.tape_capacity_bounds(t)
        bad += [f"{w} {k} bound {bd[k]} > {m}" for k, m in LIM.items() if int(bd[k]) > m]
        for r in real_rows(t):
            if int(r[C_EXEC]) == ICE:
                sl = ceil_div(r[C_QTY], r[C_PEAK]) if int(r[C_PEAK]) > 0 else 10 ** 9
                PEAK["slices"] = max(PEAK["slices"], sl)
                if sl > MAX_SLICES:
                    bad.append(f"{w} iceberg {int(r[C_OID])}: {sl} slices > {MAX_SLICES}")
        if w in ("A", "B", "CTRL"):
            eb = env_bounds(t, n)
            bad += [f"{w} env {k} bound {eb[k]} > {m}" for k, m in LIM.items() if eb[k] > m]
            PEAK["env_hop_bound"] = max(PEAK["env_hop_bound"], eb["hops"])
    bd = dict(draw.capacity_bounds)
    bad += [f"draw {k} bound {bd.get(k)} > {m}" for k, m in LIM.items() if int(bd.get(k, 0)) > m]
    return bad


wired = all(callable(globals().get(k)) for k in ("run_draw", "agent_report", "report_leg", "run_env"))
cert.check("section 6 wired (run_draw, agent_report, report_leg and run_env defined)", wired, expected=True,
           observed=wired, unit="bool", detail="" if wired else "section 6 is still the K2 stub", rule=R_ENGINE)
HAS_OWN_VIEW = callable(globals().get("own_orders_view"))

PER_DRAW, DATA_BAD, NO_DISCRIM, VALID = [], {}, {}, []
COVER = Counter()
FLAG_INSIDE_MARGIN = {}                    # seed -> I8 evidence (A and/or B), for draws inside the analytic margin
STATUS = defaultdict(Counter)              # "A:I2" / "B:I2" -> status counts over valid draws
BAT_BAD = defaultdict(list)                # "A:I2" -> seeds with a status that is not allowed
DISCARDS = Counter()
EAGER_DONE = defaultdict(list)             # "any valid draw" or eager class -> seeds with an eager comparison
EAGER_FLOAT = {"tob_max_abs": 0.0, "fill_max_abs": 0.0, "cash_inventory_max_abs": 0.0, "bit_identical_episodes": 0,
               "episodes": 0}
OWN_REPORT_ROWS = []                       # rows in the participant's random-policy report, per valid draw
OWN_ICE_HIDDEN = []                        # valid seeds whose random A episode ends with a participant iceberg's hidden part
TRADED_AS = {"buyer": [], "seller": []}    # valid seeds on which the participant trades as buyer / as seller (random A)
PART_REPORT = {"orders": Counter(), "fok": Counter(), "self_matches": 0, "own_ice_peaks_exhausted": 0}   # reported
AGENT_TRADES_SAME = Counter()              # state.agent_trades bit-identical across A/B (reported, F13)


def check_engine(seed, e, recA, recB):
    exp = {"A": e["A"], "B": e["B"], "B_is_A_relabelled": e["B_relabelled_from_A"]}
    oA, oB = observe_engine(recA), observe_engine(recB)
    obs = {"A": oA, "B": oB, "B_is_A_relabelled": oB["trades"]}
    diff = [f"{w}.{k}" for w in ("A", "B") for k, v in exp[w].items() if obs[w].get(k) != v]
    if obs["B_is_A_relabelled"] != exp["B_is_A_relabelled"]:
        diff.append("B_is_A_relabelled")
    return cert.check(f"seed {seed} (1) engine: trades and fates of A and B", not diff, seed=seed,
                      expected=canon(exp), observed=canon(obs), unit="other",
                      detail=("differs on: " + ", ".join(diff)) if diff else "identical", rule=R_ENGINE)


def check_obs_twin(seed, e, trA, trB, policy, label=""):
    vA, vB = env_view(trA), env_view(trB)
    shape_ok = vA["n_tob"] == e["n_obs"] and vA["n_steps"] == e["n_steps"] and vB["n_tob"] == e["n_obs"]
    d = env_diff(trA, trB)
    ok = shape_ok and vA == vB
    return cert.check(f"seed {seed} (2) participant-view twin, {policy} policy{label}: A and B bit-identical", ok,
                      seed=seed, expected=canon(vA), observed=canon(vB), unit="other",
                      detail=("identical" if ok else canon({"n_views_expected": e["n_obs"], "n_views": [vA["n_tob"], vB["n_tob"]],
                                                            "differences": d})), rule=R_OBS)


SURFACED = {}          # (seed, policy) -> {A iceberg oid: {f, u, k_e}}: jitted runs where the premise failed (F17)
PREMISE = Counter()    # "policy:holds" / "policy:fails" over valid draws, jitted runs


def entry_sweeps(tr, oids):
    # F17: every trade of the episode whose aggressor is one of `oids`, from the final book's trade rows.
    t = np.asarray(tr["book_final"].trades)
    out = {}
    if t.ndim != 2:
        return out
    for i in range(t.shape[0]):
        fl = _fill_from_row(t[i], i, -1)
        if int(fl.sec) == -1 or int(fl.aggressor_oid) not in oids:
            continue
        d = out.setdefault(str(int(fl.aggressor_oid)), {"f": 0, "passive_agents": []})
        d["f"] += int(fl.qty_int)
        if int(fl.passive_agent) not in d["passive_agents"]:
            d["passive_agents"].append(int(fl.passive_agent))
    for d in out.values():
        d["passive_agents"].sort()
    return out


def resting_rows(snap, oid, side):
    # Every resting row of order `oid` on `side` of a BookSnapshot, as [price_int, visible_qty, agent].
    return [[int(r[0]), int(r[1]), int(r[3])] for r in snap.resting(side) if int(r[2]) == int(oid)]


def check_surfacing(seed, e, trA, trB, policy, swA, swB, label=""):
    # (2r): the premise failed; by §4.2.1 Rule 2 (1), (4) as amended 2026-10-20 the remainder shows v1 = min(P - u, R)
    # = P - f (k_e = 0, u = f), exactly B's NON residual: the rule-implied gap is 0, the hidden remainder is h, and the
    # whole per-step view is bit-identical (finding F17).
    ice = e["ice_entry"]
    a_of_b = {str(v["oid_B"]): j for j, v in ice.items()}
    swB_A = {a_of_b[k]: v for k, v in swB.items()}
    swept = sorted(set(swA) | set(swB_A), key=int)
    k0 = min(int(ice[j]["entry_step"]) for j in swept)
    vA, vB = env_view(trA), env_view(trB)
    SA, SB = trA["book_final"], trB["book_final"]
    f_max = Q_AGENT_MAX * int(e["n_steps"])
    exp = {"n_views": [e["n_obs"], e["n_obs"]], "view": vA, "icebergs": {}}
    obs = {"n_views": [vA["n_tob"], vB["n_tob"]], "view": vB, "icebergs": {}}
    surf, implied = {}, {}
    for j in swept:
        en = ice[j]
        a = swA.get(j, {"f": 0, "passive_agents": []})
        b = swB_A.get(j, {"f": 0, "passive_agents": []})
        f = int(a["f"])
        implied[j] = s = surfacing(en, f)
        surf[j] = {"f": f, "u": s["u"], "k_e": s["k_e"]}
        exp["icebergs"][j] = {"A_entry_passive_agents": [PARTICIPANT_ID], "A_entry_fill_in_1_to_f_max_below_peak": True,
                              "B_entry_fill": f, "B_entry_passive_agents": [PARTICIPANT_ID],
                              "visible_gap_A_minus_B": s["visible_gap"], "hidden_A": s["hidden_A"],
                              "A_resting_rows_price": [s["rest_price_A"]]}
        obs["icebergs"][j] = {"A_entry_passive_agents": a["passive_agents"],
                              "A_entry_fill_in_1_to_f_max_below_peak": 1 <= f <= f_max < int(en["peak"]),
                              "B_entry_fill": int(b["f"]), "B_entry_passive_agents": b["passive_agents"],
                              "visible_gap_A_minus_B": int(SA.visible_qty(int(j), en["side"]))
                              - int(SB.visible_qty(int(en["oid_B"]), en["side"])),
                              "hidden_A": int(SA.hidden_qty(int(j))),
                              "A_resting_rows_price": [r[0] for r in resting_rows(SA, j, en["side"])]}
    diff = [k for k in exp if k != "icebergs" and obs[k] != exp[k]]
    diff += [f"iceberg {j}" for j in swept if obs["icebergs"][j] != exp["icebergs"][j]]
    if not label:
        SURFACED[(seed, policy)] = surf
    detail = canon({"differs_on": diff, "f_max": f_max, "first_entry_step": k0, "rule_implied": implied,
                    "view_differences": env_diff(trA, trB)})
    return cert.check(f"seed {seed} (2r) rule-implied display, {policy} policy{label}: a tape iceberg's entry swept "
                      f"the participant's resting order; its remainder shows v1 = min(P - u, R) = P - f, as B's residual "
                      f"(§4.2.1 Rule 2 (1), (4), amended 2026-10-20), and the twin stays bit-identical",
                      not diff, seed=seed, expected=canon(exp), observed=canon(obs), unit="other", detail=detail,
                      rule=R_SURF)


def check_obs_pair(seed, e, trA, trB, policy, label=""):
    # F17: the premise check, then (2) where it holds and (2r) where it fails.
    ice = e["ice_entry"]
    swA = guarded(f"seed {seed} premise check A/{policy}{label}", seed, entry_sweeps, trA, {int(j) for j in ice})
    swB = guarded(f"seed {seed} premise check B/{policy}{label}", seed, entry_sweeps, trB,
                  {int(v["oid_B"]) for v in ice.values()})
    if swA is None or swB is None:
        return False
    holds = not swA and not swB
    if not label:
        PREMISE[f"{policy}:{'holds' if holds else 'fails'}"] += 1
    if holds:
        return check_obs_twin(seed, e, trA, trB, policy, label)
    return check_surfacing(seed, e, trA, trB, policy, swA, swB, label)


def check_control(seed, trB, trC):
    vB, vC = env_view(trB), env_view(trC)
    tob_steps = [k for k, (x, y) in enumerate(zip(vB["tob"], vC["tob"])) if x != y]
    both = [k for k in tob_steps if k < min(len(vB["lob"]), len(vC["lob"])) and vB["lob"][k] != vC["lob"][k]]
    ok = bool(both)
    d = [x for x in env_diff(trB, trC, "B", "CTRL") if x.get("field") in ("tob", "lob")]
    return cert.check(f"seed {seed} (3) positive control: one visible tick changes the noop top_of_book (and "
                      f"get_anonymous_lob on that step)", ok, seed=seed,
                      expected="tob(CTRL) != tob(B) on >= 1 step, and lob(CTRL) != lob(B) on that step",
                      observed=canon({"tob_steps_differing": tob_steps, "of_which_lob_differs": both,
                                      "B": vB["tob"], "CTRL": vC["tob"]}),
                      unit="other", detail=canon(d) if ok else "top_of_book did not see the visible change",
                      rule=R_CTRL)


def check_partition(seed, e, raw):
    obs_legs, n_rows = {}, 0
    for a in e["report_ids"]:
        rows = raw.get(a)
        if rows is None:
            obs_legs[str(a)] = None
            continue
        n_rows += len(rows)
        got = guarded(f"seed {seed} report_leg (agent {a})", seed, lambda: sorted(
            [int(x) for x in report_leg(r, a)] for r in rows))
        obs_legs[str(a)] = got
    union = sorted(l for v in obs_legs.values() if v for l in v)
    exp = {"reports": e["reports"], "union": e["union"], "n_report_rows": e["n_report_rows"]}
    obs = {"reports": obs_legs, "union": union, "n_report_rows": n_rows}
    diff = [f"reports[{a}]" for a in e["reports"] if obs_legs.get(a) != e["reports"][a]]
    diff += [k for k in ("union", "n_report_rows") if obs[k] != exp[k]]
    return cert.check(f"seed {seed} (4) trade-report partition and union", not diff, seed=seed, expected=canon(exp),
                      observed=canon(obs), unit="other",
                      detail=("differs on: " + ", ".join(diff)) if diff else "identical", rule=R_PART)


def check_report_twin(seed, e, rawA, rawT):
    exp = {str(a): None if rawA.get(a) is None else plain(rawA[a]) for a in e["observers"]}
    obs = {str(a): None if rawT.get(a) is None else plain(rawT[a]) for a in e["observers"]}
    diff = [a for a in exp if exp[a] is None or exp[a] != obs[a]]
    return cert.check(f"seed {seed} (5) trade-report twin: each observer's report in A and B@a byte-identical",
                      not diff, seed=seed, expected=canon(exp), observed=canon(obs), unit="other",
                      detail=("differs for observers: " + ", ".join(diff)) if diff else "identical", rule=R_ANON)


def check_ownership(seed, e, recA):
    obs = observe_ownership(recA, e["ice_oids_A"])
    diff = [j for j in e["ownership"] if obs.get(j) != e["ownership"][j]]
    return cert.check(f"seed {seed} (6) ownership: each iceberg under its owner, full remaining quantity held",
                      not diff, seed=seed, expected=canon(e["ownership"]), observed=canon(obs), unit="other",
                      detail=("differs for icebergs: " + ", ".join(diff)) if diff else "identical", rule=R_OWN)


_OF = {f: i for i, f in enumerate(OMIEOrder._fields)}   # field positions of a get_agent_orders row


def own_from_book(tr):
    # (6''), expected: the participant's resting rows on the episode's final book, {oid: [visible + hidden]}.
    s, out = tr["book_final"], {}
    for side in (BUY, SELL):
        for r in s.resting(side):
            if int(r[3]) == PARTICIPANT_ID:
                out.setdefault(str(int(r[2])), []).append(int(r[1]) + int(s.hidden_qty(int(r[2]))))
    return out


def own_from_reader(tr):
    # (6''), observed: get_agent_orders(final book, AGENT_TRADER_ID) as {oid: [quantity ticks]} and the owners shown.
    out, owners = {}, set()
    for o in tr["own_orders"]:
        out.setdefault(str(int(o[_OF["order_id"]])), []).append(int(encode_qty(float(o[_OF["quantity"]]), CONFIG)))
        owners.add(int(o[_OF["agent_id"]]))
    return out, sorted(owners)


def anon_at_own_icebergs(tr):
    # F23: at each price where a participant iceberg rests with a hidden part, the visible sum there (from the book)
    # and the quantity get_anonymous_lob shows at that price on the last step (ticks; None = level not shown).
    s, lob, exp, obs = tr["book_final"], tr["lob"][-1], {}, {}
    for side in (BUY, SELL):
        for r in s.resting(side):
            if int(r[3]) != PARTICIPANT_ID or int(s.hidden_qty(int(r[2]))) <= 0:
                continue
            key = f"{side_name(side)}@{int(r[0])}"
            exp[key] = sum(int(x[1]) for x in s.resting(side) if int(x[0]) == int(r[0]))
            levels = lob["bids" if side == BUY else "asks"]
            shown = [int(encode_qty(float(q), CONFIG)) for pr, q in levels
                     if int(encode_price(float(pr), CONFIG)) == int(r[0])]
            obs[key] = shown[0] if len(shown) == 1 else (shown or None)
    return exp, obs


def check_own_participant(seed, trA, trB):
    # (6''): the participant's own-orders reader in the random episodes (finding F6), and F23 on the anonymous book.
    rA, ownersA = own_from_reader(trA)
    anon_exp, anon_obs = anon_at_own_icebergs(trA)
    exp = {"A_own_orders": own_from_book(trA), "A_owners_shown": [PARTICIPANT_ID] if trA["own_orders"] else [],
           "B_own_orders_bit_identical_to_A": True, "A_anonymous_book_at_own_icebergs_shows_visible_only": anon_exp}
    obs = {"A_own_orders": rA, "A_owners_shown": ownersA,
           "B_own_orders_bit_identical_to_A": trB["own_orders"] == trA["own_orders"],
           "A_anonymous_book_at_own_icebergs_shows_visible_only": anon_obs}
    diff = [k for k in exp if obs[k] != exp[k]]
    return cert.check(f"seed {seed} (6″) ownership (participant): get_agent_orders shows the participant exactly its "
                      f"resting orders at full remaining quantity, bit-identical in A and B; the anonymous book shows "
                      f"its icebergs only by their visible slice", not diff, seed=seed, expected=canon(exp),
                      observed=canon(obs), unit="other",
                      detail=("differs on: " + ", ".join(diff) + " | B: " + canon(trB["own_orders"])) if diff
                      else "identical", rule=R_OWNP)


def participant_stats(seed, draw, tr):
    # Non-vacuity inputs and reported counts from the random A episode (jitted).
    s, acts = tr["book_final"], draw.params["actions"]
    if any(int(r[3]) == PARTICIPANT_ID and int(s.hidden_qty(int(r[2]))) > 0 for sd in (BUY, SELL) for r in s.resting(sd)):
        OWN_ICE_HIDDEN.append(seed)
    t = np.asarray(s.trades)
    buy = sell = False
    passive_by_oid = Counter()
    for i in range(t.shape[0]):
        fl = _fill_from_row(t[i], i, -1)
        if int(fl.sec) == -1:
            continue
        buy |= int(fl.buyer_agent) == PARTICIPANT_ID
        sell |= int(fl.seller_agent) == PARTICIPANT_ID
        if int(fl.buyer_agent) == PARTICIPANT_ID and int(fl.seller_agent) == PARTICIPANT_ID:
            PART_REPORT["self_matches"] += 1
        if int(fl.passive_agent) == PARTICIPANT_ID:
            passive_by_oid[int(fl.passive_oid)] += int(fl.qty_int)
    if buy:
        TRADED_AS["buyer"].append(seed)
    if sell:
        TRADED_AS["seller"].append(seed)
    for k, (act, sd, ex, _p, _q, pk) in enumerate(acts):
        if not int(act):
            continue
        PART_REPORT["orders"][f"{EXEC_NAME[int(ex)]} {side_name(sd)}"] += 1
        if int(ex) == FOK:
            f = tr["fills"][k]
            PART_REPORT["fok"]["ignored (after done)" if not bool(f[".submitted"]) else
                               ("filled" if bool(f[".submit.traded"]) else "killed")] += 1
        if int(ex) == ICE and int(pk) > 0:
            PART_REPORT["own_ice_peaks_exhausted"] += passive_by_oid[int(AGENT_ORDER_ID_BASE) + k] // int(pk)


def check_twin_kept(seed, e, trs):
    # (7): hidden intact at the end of every A episode, none in B/CTRL, no capacity flag in any episode.
    exp, obs = {}, {}
    for (w, pol), tr in trs.items():
        oids = e["ice_oids_A"] if w == "A" else e["oids"][w]
        h = env_hidden(tr, oids)
        hid = dict(e["hidden_A"]) if w == "A" else {}
        if w == "A":   # an iceberg of (2r) holds the rule-implied R - v1 (= h here, K2 r4; F17)
            for j, sv in SURFACED.get((seed, pol), {}).items():
                hid[j] = surfacing(e["ice_entry"][j], sv["f"])["hidden_A"]
        exp[f"{w}/{pol}"] = {"hidden": hid, "book_full": False, "truncated": False,
                             "trade_rows_used_le": ENVELOPE.max_trades}
        f = env_flags(tr)
        PEAK["env_trade_rows"] = max(PEAK["env_trade_rows"], f["trade_rows_used"])
        obs[f"{w}/{pol}"] = {"hidden": h if w == "A" else {k: v for k, v in h.items() if v},
                             "book_full": f["book_full"], "truncated": f["truncated"],
                             "trade_rows_used_le": ENVELOPE.max_trades if f["trade_rows_used"] <= ENVELOPE.max_trades
                             else f["trade_rows_used"]}
    diff = [k for k in exp if obs.get(k) != exp[k]]
    return cert.check(f"seed {seed} (7) the twin stayed a twin: hidden remainders intact in A, none in B/CTRL, "
                      f"no capacity flag", not diff, seed=seed, expected=canon(exp), observed=canon(obs),
                      unit="other", detail=("differs on: " + ", ".join(diff)) if diff else "identical",
                      rule=R_TWIN + "; " + R_CAP)


def eager_compare(seed, draw, e, recA, rawA, trs):
    # Eager against jitted: integer fields equal; the participant-view twin holds in eager mode too.
    diff = []
    recA_e = guarded(f"seed {seed} engine A (eager)", seed, run_draw, draw, "A", jit=False)
    diff += ["engine A run missing"] if recA_e is None else [f"engine A: {x}" for x in compare_records(recA_e, recA)]
    for a in e["report_ids"]:
        r = guarded(f"seed {seed} report A agent {a} (eager)", seed, agent_report, draw, "A", a, jit=False)
        if r is None or rawA.get(a) is None or plain(r) != plain(rawA[a]):
            diff.append(f"report A agent {a}")
    eager_tr = {}
    for w in ("A", "B"):
        for pol in POLICIES:
            tr = guarded(f"seed {seed} env {w}/{pol} (eager)", seed, run_env, draw, w, pol, jit=False)
            eager_tr[(w, pol)] = tr
            tj = trs.get((w, pol))
            if tr is None or tj is None:
                diff.append(f"env {w}/{pol} missing")
                continue
            if [np.asarray(d).tolist() for d in tr["dones"]] != [np.asarray(d).tolist() for d in tj["dones"]]:
                diff.append(f"env {w}/{pol}: dones")
            if plain(tr["own_report"]) != plain(tj["own_report"]):
                diff.append(f"env {w}/{pol}: own_report")
            if tr["own_orders"] != tj["own_orders"]:
                diff.append(f"env {w}/{pol}: own_orders")
            if snap_key(tr["book_final"]) != snap_key(tj["book_final"]):
                diff.append(f"env {w}/{pol}: final book")
            if [tree_digest(x) for x in tr["lob"]] != [tree_digest(x) for x in tj["lob"]]:
                diff.append(f"env {w}/{pol}: get_anonymous_lob")
            # integer (and boolean) fields of top_of_book and the FillReport are asserted; float fields are reported
            for key, fkey in (("tob", "tob_max_abs"), ("fills", "fill_max_abs")):
                if len(tr[key]) != len(tj[key]):
                    diff.append(f"env {w}/{pol}: {key} length")
                    continue
                for k, (x, y) in enumerate(zip(tr[key], tj[key])):
                    for name in sorted(set(x) | set(y)):
                        a, b = np.asarray(x.get(name)), np.asarray(y.get(name))
                        if a.dtype.kind == "f" and b.dtype.kind == "f" and a.shape == b.shape:
                            EAGER_FLOAT[fkey] = max(EAGER_FLOAT[fkey], float(np.max(np.abs(
                                a.astype(np.float64) - b.astype(np.float64)), initial=0.0)))
                        elif a.dtype != b.dtype or a.shape != b.shape or a.tobytes() != b.tobytes():
                            diff.append(f"env {w}/{pol}: {key} step {k} {name}")
            for key in ("cash", "inventory"):
                try:
                    EAGER_FLOAT["cash_inventory_max_abs"] = max(EAGER_FLOAT["cash_inventory_max_abs"], float(np.max(
                        np.abs(np.asarray(tr[key], dtype=np.float64) - np.asarray(tj[key], dtype=np.float64)))))
                except Exception:  # noqa: BLE001
                    pass
            EAGER_FLOAT["episodes"] += 1
            if env_view(tr) == env_view(tj):
                EAGER_FLOAT["bit_identical_episodes"] += 1
    for t in ["any valid draw"] + list(e["draw_classes"]):
        EAGER_DONE[t].append(seed)
    ok = not diff
    cert.check(f"eager == jitted (seed {seed}): integer fields of the A record, the A reports and the four episodes",
               ok, kind="eager_jit", seed=seed, expected=True, observed=ok, unit="bool",
               detail="identical" if ok else ", ".join(diff), rule=R_EAGER)
    for pol in POLICIES:
        if eager_tr.get(("A", pol)) is not None and eager_tr.get(("B", pol)) is not None:
            check_obs_pair(seed, e, eager_tr[("A", pol)], eager_tr[("B", pol)], pol, " (eager)")


def battery_statuses(world, rep, seed):
    for r in rep.results:
        STATUS[f"{world}:{r.check_id}"][r.status] += 1
        if r.check_id in MUST_PASS + NO_FAIL and r.status not in SATISFIED:
            BAT_BAD[f"{world}:{r.check_id}"].append(seed)


def run_all():
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        draw = DRAWN[seed] if seed in DRAWN else prepare(seed)
        if draw is None:
            cert.discard(seed, f"builder refused the seed: {REFUSED.get(seed)}")
            DISCARDS["builder refused"] += 1
            seed += 1
            continue
        bad = precondition(draw)
        if bad:
            DATA_BAD[str(seed)] = bad
            cert.check(f"seed {seed}: draw is as section 4 describes", False, seed=seed, expected=[], observed=bad,
                       unit="other", rule=R_DATA)
            seed += 1
            continue
        mb = margin_breaches(draw)
        if mb:
            cert.discard(seed, "pre-run capacity margin: " + canon(mb))
            DISCARDS["pre-run margin"] += 1
            seed += 1
            continue
        e = expectation(draw)
        recA = guarded(f"seed {seed} engine A (jit)", seed, run_draw, draw, "A", jit=True)
        recB = guarded(f"seed {seed} engine B (jit)", seed, run_draw, draw, "B", jit=True)
        if recA is None or recB is None:
            seed += 1
            continue
        repA = run_battery(recA, expects_trade=e["expects_trade"])
        repB = run_battery(recB, expects_trade=e["expects_trade"])
        i8 = {w: next(r for r in rep.results if r.check_id == "I8") for w, rep in (("A", repA), ("B", repB))}
        if i8["B"].status == "invalid":
            FLAG_INSIDE_MARGIN[str(seed)] = {w: dict(r.evidence) for w, r in i8.items()}
            cert.discard(seed, "I8 invalid on the twin B")
            DISCARDS["I8 invalid"] += 1
            seed += 1
            continue
        if not cert.add_battery(repA, seed=seed):
            FLAG_INSIDE_MARGIN[str(seed)] = {w: dict(r.evidence) for w, r in i8.items()}
            DISCARDS["I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F10)
            continue
        valid += 1
        VALID.append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": CONSTRUCTION})
        battery_statuses("A", repA, seed)
        battery_statuses("B", repB, seed)
        cert.record_capacity({**recA.observed_capacity, "hops": None})   # hops: unobserved by the engine
        cert.record_capacity({**recB.observed_capacity, "hops": None})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["trades"] = max(PEAK["trades"], len(recA.fills), len(recB.fills))
        for c in e["classes"]:
            COVER[c] += 1
        und = [n for n in WRONG_REPORTS if not e["disc"][n]]
        if und:
            NO_DISCRIM[str(seed)] = und
        # the reports
        rawA = {a: guarded(f"seed {seed} report A agent {a} (jit)", seed, agent_report, draw, "A", a, jit=True)
                for a in e["report_ids"]}
        rawT = {a: guarded(f"seed {seed} report {w} agent {a} (jit)", seed, agent_report, draw, w, a, jit=True)
                for a, w in observer_worlds(draw).items()}
        # the episodes
        trs = {}
        for w, pol in (("A", "noop"), ("B", "noop"), ("CTRL", "noop"), ("A", "random"), ("B", "random")):
            trs[(w, pol)] = guarded(f"seed {seed} env {w}/{pol} (jit)", seed, run_env, draw, w, pol, jit=True)
        ok = [check_engine(seed, e, recA, recB)]
        for pol in POLICIES:
            if trs[("A", pol)] is not None and trs[("B", pol)] is not None:
                ok.append(check_obs_pair(seed, e, trs[("A", pol)], trs[("B", pol)], pol))
            else:
                ok.append(False)
        if trs[("B", "noop")] is not None and trs[("CTRL", "noop")] is not None:
            ok.append(check_control(seed, trs[("B", "noop")], trs[("CTRL", "noop")]))
        else:
            ok.append(False)
        ok.append(check_partition(seed, e, rawA))
        ok.append(check_report_twin(seed, e, rawA, rawT))
        ok.append(check_ownership(seed, e, recA))
        if all(tr is not None for tr in trs.values()):
            ok.append(check_twin_kept(seed, e, trs))
        else:
            ok.append(False)
        if HAS_OWN_VIEW:
            view = {str(a): guarded(f"seed {seed} own_orders_view agent {a}", seed, own_orders_view, draw, "A", a,
                                    jit=True) for a in e["observers"]}
            view = {a: None if v is None else {str(int(k)): int(x) for k, x in dict(v).items()} for a, v in view.items()}
            ok.append(cert.check(f"seed {seed} (6') own-orders view shows each owner its full remaining quantity",
                                 view == e["own_orders"], seed=seed, expected=canon(e["own_orders"]),
                                 observed=canon(view), unit="other", rule=R_OWN))
        tr_r = trs.get(("A", "random"))
        if tr_r is not None:
            OWN_REPORT_ROWS.append(len(tr_r["own_report"]))
            guarded(f"seed {seed} participant statistics", seed, participant_stats, seed, draw, tr_r)
        if tr_r is not None and trs.get(("B", "random")) is not None:
            ok.append(check_own_participant(seed, tr_r, trs[("B", "random")]))
        else:
            ok.append(False)
        for pol in POLICIES:
            ta, tb = trs.get(("A", pol)), trs.get(("B", pol))
            if ta is not None and tb is not None:
                try:
                    AGENT_TRADES_SAME[digest(ta["agent_trades"]) == digest(tb["agent_trades"])] += 1
                except Exception:  # noqa: BLE001
                    AGENT_TRADES_SAME["unreadable"] += 1
        PER_DRAW.append({"seed": seed, "classes": e["draw_classes"], "ok": all(ok)})
        if valid <= EAGER_DRAWS or any(c not in EAGER_DONE for c in e["draw_classes"]):
            eager_compare(seed, draw, e, recA, rawA, trs)
        seed += 1
    return valid


N_VALID = run_all() if wired else 0

# ---- aggregate assertions (every list in full) ----------------------------------------------------------------
cert.check("dataset: every draw is as section 4 describes (precondition)", wired and not DATA_BAD,
           expected={}, observed=DATA_BAD, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{N_VALID} valid draws", rule=R_DATA)
cert.check("discrimination: each wrong report (every trade, buy legs only, sell legs only) misses the rule on every "
           "valid draw", wired and N_VALID > 0 and not NO_DISCRIM, expected={}, observed=NO_DISCRIM, unit="other",
           detail=f"valid draws: {N_VALID}", rule=R_DISC)
for _w in ("A", "B"):
    for _cid in MUST_PASS:
        _k = f"{_w}:{_cid}"
        cert.check(f"{_cid} passes (not_applicable = vacuously satisfied) on world {_w} of every valid draw",
                   wired and N_VALID > 0 and not BAT_BAD[_k],
                   expected=[], observed=BAT_BAD[_k], unit="other",
                   detail=f"status counts over {N_VALID} valid draws: {dict(STATUS[_k])}; not_applicable on "
                          f"{STATUS[_k]['not_applicable']} draws (vacuously satisfied)", rule=R_BAT)
    for _cid in NO_FAIL:
        _k = f"{_w}:{_cid}"
        cert.check(f"{_cid} never fails on world {_w} of a valid draw (not_applicable allowed)",
                   wired and N_VALID > 0 and not BAT_BAD[_k], expected=[], observed=BAT_BAD[_k], unit="other",
                   detail=f"status counts over {N_VALID} valid draws: {dict(STATUS[_k])}; not_applicable on "
                          f"{STATUS[_k]['not_applicable']} draws (reported, not asserted)", rule=R_BAT)
cert.check("no capacity flag (truncated, book_full, saturated ring) on an engine world inside the analytic margin (F10)",
           wired and not FLAG_INSIDE_MARGIN, expected={}, observed=FLAG_INSIDE_MARGIN, unit="other", rule=R_CAP)
cert.check("hop bound <= 160 on every valid draw (largest reported: tape bound, env-side bound)",
           wired and N_VALID > 0 and PEAK["hop_bound"] <= ENVELOPE.max_hops and PEAK["env_hop_bound"] <= ENVELOPE.max_hops,
           expected=f"<= {ENVELOPE.max_hops}", observed=[PEAK["hop_bound"], PEAK["env_hop_bound"]], unit="count",
           rule=R_CAP)
cert.check("trades <= 80 on every valid draw (largest observed: engine fills, env trade rows)", wired and N_VALID > 0
           and PEAK["trades"] <= ENVELOPE.max_trades and PEAK["env_trade_rows"] <= ENVELOPE.max_trades,
           expected=f"<= {ENVELOPE.max_trades}", observed=[PEAK["trades"], PEAK["env_trade_rows"]], unit="count",
           rule=R_CAP)
cert.check("every iceberg within ceil(total / peak) <= 64 slices (largest drawn)", wired and N_VALID > 0
           and PEAK["slices"] <= MAX_SLICES, expected=f"<= {MAX_SLICES}", observed=PEAK["slices"], unit="count",
           rule=R_CAP)
_got = {c: COVER[c] for c in ALL_CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid draws cover every class (section 3)", wired and not _missing,
           expected={c: ">= 1" for c in ALL_CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)
cert.check("non-vacuity: the participant's own trade report is non-empty (random policy) on at least one valid draw",
           wired and any(n > 0 for n in OWN_REPORT_ROWS), expected=">= 1 draw", unit="count",
           observed=sum(1 for n in OWN_REPORT_ROWS if n > 0),
           detail=f"rows per draw: min {min(OWN_REPORT_ROWS, default=0)}, max {max(OWN_REPORT_ROWS, default=0)}",
           rule=R_OBS)

_c3 = sorted({s for (s, pol), sv in SURFACED.items() if pol == "random" and any(v["u"] > 0 for v in sv.values())})
cert.check("non-vacuity: display C3 is exercised -- a tape iceberg's entry sweeps the participant's resting order with "
           "u > 0 (premise fails, random policy, jitted) on at least one valid draw", wired and len(_c3) > 0,
           expected=">= 1 draw", observed=len(_c3), unit="count",
           detail=f"seeds {_c3}; per seed/policy, iceberg -> {{f, u, k_e}}: "
                  f"{ {f'{s}/{pol}': sv for (s, pol), sv in sorted(SURFACED.items())} }", rule=R_SURF)

cert.check("non-vacuity: a participant iceberg rests with a hidden part at the end of the random episode on at least "
           "one valid draw, so (6″) shows a hidden part", wired and len(OWN_ICE_HIDDEN) > 0, expected=">= 1 draw",
           observed=len(OWN_ICE_HIDDEN), unit="count", detail=f"seeds {OWN_ICE_HIDDEN}", rule=R_OWNP)
for _side in ("buyer", "seller"):
    cert.check(f"non-vacuity: the participant trades as {_side} (random policy) on at least one valid draw",
               wired and len(TRADED_AS[_side]) > 0, expected=">= 1 draw", observed=len(TRADED_AS[_side]), unit="count",
               detail=f"seeds {TRADED_AS[_side]}", rule=R_OBS)

# ---- eager: at least one comparison, and one per eager class (absence = failure) -------------------------------
for _t in ("any valid draw",) + EAGER_CLASSES:
    if not EAGER_DONE.get(_t):
        cert.check(f"eager == jitted for '{_t}'", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"no eager comparison was run for '{_t}'", rule=R_EAGER)

# ---- reports (not asserted) ------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_DRAW if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_DRAW)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"seed {r['seed']}" for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws: {N_VALID}; per class: {dict(COVER)}")
print(f"discards per reason: {dict(DISCARDS)}")
_pf_seeds = sorted({s for s, _pol in SURFACED})
print(f"premise check (F17), jitted runs over valid draws: {dict(PREMISE)}")
print(f"premise failed -> (2r) asserted, draw kept (no discard): {len(_pf_seeds)} of {N_VALID} valid draws "
      f"({100.0 * len(_pf_seeds) / max(N_VALID, 1):.1f} %), seeds {_pf_seeds}; per seed/policy, iceberg -> {{f, u, k_e}}: "
      f"{ {f'{s}/{pol}': f for (s, pol), f in sorted(SURFACED.items())} }")
print(f"eager comparisons run: { {k: v for k, v in EAGER_DONE.items()} }")
print(f"eager vs jitted floats of top_of_book, FillReport, cash and inventory (reported, V2): {EAGER_FLOAT}")
print(f"participant orders by restriction and side (random policy, valid draws): {dict(PART_REPORT['orders'])}")
print(f"participant FOK orders (random A episodes): {dict(PART_REPORT['fok'])}")
print(f"participant self-matches (F8, reported): {PART_REPORT['self_matches']}; participant iceberg peaks exhausted "
      f"(floor(passive fill / peak), i.e. slices regenerated or exhausted, random A episodes): "
      f"{PART_REPORT['own_ice_peaks_exhausted']}")
print(f"participant traded as buyer on {len(TRADED_AS['buyer'])} and as seller on {len(TRADED_AS['seller'])} valid draws; "
      f"ended with a hidden participant iceberg on {len(OWN_ICE_HIDDEN)}")
print(f"battery status counts over valid draws: { {k: dict(v) for k, v in STATUS.items()} }")
print(f"state.agent_trades bit-identical across A/B episodes (reported, F13): {dict(AGENT_TRADES_SAME)}")
print("F24 (recorded, not asserted): neither top_of_book nor get_anonymous_lob returns the LOB timestamps of "
      "section 9.2 'LOB per contract' -- a representation gap, not a leak")
print(f"owner-facing own-orders view exposed by section 6 (F6): {HAS_OWN_VIEW}"
      + ("" if HAS_OWN_VIEW else " -- recorded: no owner-facing view; the ownership side-condition is certified at "
                                 "the engine and in the twin structure only"))
print(f"analytic peaks over valid draws: {PEAK}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))